# ComfyUI instance on Colab

ColabのGPUでComfyUIを起動し、[Comfy Desktop](https://github.com/Comfy-Org/Comfy-Desktop) の **Remote Connection**、または通常のブラウザーから接続します。モデルと環境はGoogle Driveの `ComfyUI-Colab/` に保存します。

**使い方:** ランタイムのタイプでGPUを選び、**1 → 2 → 3 → 6-1〜6-3 のいずれか** の順に実行します。入力欄やチェックボックスを設定してから、そのセルを実行してください。チェックボックスのある操作は、ONにして実行したときだけ行われます。

**前提:** Driveへの接続はDrive全体への許可になります。個人データを置かない専用のGoogleアカウントを使い、信頼できるカスタムノードとモデルだけを導入してください（末尾の「セキュリティについて」）。[Colabの利用条件](https://research.google.com/colaboratory/faq.html) に従い、有料プランでの利用を前提とします。


In [ ]:
#@title 0. 更新確認
#@markdown ComfyUIの最新リリースを確認します。ONにすると、1の `COMFY_REF` の代わりに最新版を使います。
USE_LATEST_COMFYUI = False #@param {type:"boolean"}
import re, subprocess
found = subprocess.run(["git", "ls-remote", "--tags", "--refs", "https://github.com/Comfy-Org/ComfyUI.git", "v*"],
                       capture_output=True, text=True, timeout=60)
releases = [tuple(map(int, tag.groups())) for tag in
            (re.search(r"refs/tags/v(\d+)\.(\d+)\.(\d+)$", line) for line in found.stdout.splitlines()) if tag]
if found.returncode or not releases:
    raise RuntimeError("最新リリースを取得できませんでした。しばらくしてから実行し直してください。")
LATEST_COMFY_REF = "v" + ".".join(map(str, max(releases)))
print("ComfyUIの最新リリース:", LATEST_COMFY_REF)
# Cell 1 sets COMFY_REF from its form, so it takes the choice made here from this name.
COMFY_REF_OVERRIDE = LATEST_COMFY_REF if USE_LATEST_COMFYUI else ""
if USE_LATEST_COMFYUI:
    COMFY_REF = LATEST_COMFY_REF
    print("COMFY_REFを最新リリースにしました。続けて1を実行してください。")
else:
    print("COMFY_REFは1の入力値を使います。最新リリースを使う場合は、USE_LATEST_COMFYUIをONにして実行してください。")


In [ ]:
#@title 1. 設定
#@markdown Driveの保存先フォルダー（マイドライブからの相対パス）。同じフォルダーを複数のランタイムで同時に使わないでください。
DRIVE_FOLDER_NAME = "ComfyUI-Colab" #@param {type:"string"}
#@markdown ComfyUIのバージョン（リリースタグ）。新しい環境を作るときと、11で更新するときに使われます。
COMFY_REF = "v0.39.0" #@param {type:"string"}
#@markdown 環境を自動保存する間隔（秒、60以上）。
CHECKPOINT_SECONDS = 120 #@param {type:"integer"}
#@markdown Driveに保存した環境（カスタムノード・設定）を復元する。OFFにすると新しい環境で始めます。
RESTORE_SAVED_ENVIRONMENT = True #@param {type:"boolean"}
#@markdown 追加で必要なOSパッケージ（空白区切り、任意）。
APT_PACKAGES = "" #@param {type:"string"}
if globals().get("COMFY_REF_OVERRIDE"):
    COMFY_REF = COMFY_REF_OVERRIDE
print("COMFY_REF:", COMFY_REF)


In [ ]:
#@title 2. 実行コードを展開
import subprocess, sys
from pathlib import Path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp>=3.11,<4", "requests", "PyJWT[crypto]>=2.8,<3"], check=True)
Path('/content/comfy_colab_runtime.py').write_text('"""Self-contained runtime embedded in the ComfyUI instance on Colab notebooks."""\nfrom __future__ import annotations\n\nimport argparse\nimport asyncio\nimport hashlib\nimport hmac\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport secrets\nimport shutil\nimport signal\nimport stat\nimport subprocess\nimport sys\nimport tarfile\nimport threading\nimport tempfile\nimport time\nfrom urllib.parse import urlsplit, urljoin, quote, unquote, parse_qs\n\nBASE = Path("/content/comfy-session")\nPRIVATE = Path("/content/.comfy-private")\nDRIVE_MOUNT = PRIVATE / "drive"\nCONFIG = PRIVATE / "session.json"\nCOOKIE = "__Host-comfy_colab"\n# What a session adds to the saved settings, and stopping takes away again.\nSESSION_KEYS = ("mode", "cookie_secret", "remote_url", "origins", "access")\nPAIRING_PATH = "/_colab/pair"\nPAIRING_SECONDS = 600\nPAIRING_ATTEMPTS = 5\nCOOKIE_SECONDS = 24 * 3600\nACCESS_KEYS_REFRESH = 30\nEXCLUDED = {"models", "output", "temp", "input"}\nAPP_USER = "comfy"\nSETPRIV = "/usr/bin/setpriv"\n# Bounds on what root accepts from, and how long it waits for, commands run as that user.\nAPP_TIMEOUT = 3600\nAPP_OUTPUT_LIMIT = 16 * 1024 ** 2\nARCHIVE_LIMIT = 4 * 1024 ** 3\nMODEL_FOLDERS = ("checkpoints", "diffusion_models", "unet", "text_encoders", "clip", "clip_vision", "vae",\n                 "loras", "controlnet", "upscale_models", "embeddings", "audio_encoders", "model_patches")\nCLOUDFLARED_VERSION = "2026.10.0"\nCLOUDFLARED_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"\nMESSAGES = {\n    \'common.none\': \'なし\',\n    \'common.unknown\': \'不明\',\n    \'common.present\': \'あり\',\n    \'common.yes\': \'あり\',\n    \'common.no\': \'なし\',\n    \'path.use_slashes\': \'相対パスには / を使い、ドライブ名を含めないでください。\',\n    \'path.unsafe\': \'安全な相対パスを指定してください。\',\n    \'path.outside_folder\': \'保存先が指定フォルダー外を指しています。\',\n    \'path.symlink\': \'保存先にシンボリックリンクがあります。\',\n    \'command.running\': \'実行中: \',\n    \'command.failed\': \'{stage} が失敗しました: {failure}\',\n    \'command.exit_code\': \'{stage} が終了コード {code} で失敗しました。直前の出力を確認してください。\',\n    \'command.done\': \'完了: {stage}（{seconds:.0f}秒）\',\n    \'command.timed_out\': \'時間内に終わりませんでした\',\n    \'command.output_limit\': \'出力が上限を超えました\',\n    \'user.processes_remain\': \'ComfyUI用ユーザーのプロセスを終了できませんでした。ランタイムを削除して接続し直してください。\',\n    \'user.setpriv_missing\': \'{setpriv} がありません。権限を下げてComfyUIを実行できないため中止します。\',\n    \'app.no_outputs\': \'回収する生成ファイルがありません。\',\n    \'app.failed\': \'{action} に失敗しました: {failure}\',\n    \'app.failed_detail\': \'{action} に失敗しました: \',\n    \'venv.pip_fallback\': \'ノートブック側のpipから仮想環境へ導入できませんでした。OSのensurepipを使います。\',\n    \'venv.ensurepip_missing\': \'Python {version}のensurepipが不足しています。{package}をインストールします。\',\n    \'venv.package_failed\': \'{package}をインストールできませんでした。Colabで apt-cache policy {package} を実行し、提供バージョンとAPTのエラー出力を確認してください。\',\n    \'venv.ensurepip_unavailable\': \'{package}のインストール後も {python} からensurepipを参照できません。Pythonのインストール先とAPTパッケージを確認してください。\',\n    \'drive.needs_root\': \'Driveの保護にはColabのroot権限を持つLinuxランタイムが必要です。\',\n    \'drive.private_symlink\': \'保護ディレクトリにシンボリックリンクがあります。\',\n    \'drive.private_owner\': \'保護ディレクトリの所有者がrootではありません。\',\n    \'drive.mounted_elsewhere\': \'別の場所にDriveがマウントされています。ランタイムを削除し、このノートブックだけでマウントしてください。\',\n    \'drive.no_mydrive\': \'Google DriveのMyDriveを確認できません。\',\n    \'drive.not_mounted\': \'Driveがマウントされていません。セットアップから実行してください。\',\n    \'models.synced\': \'Driveからモデルを反映しました: {copied}ファイル\',\n    \'environment.archive_special_files\': \'環境アーカイブにリンクまたは特殊ファイルが含まれています。\',\n    \'environment.archive_needs_data_filter\': \'安全なアーカイブ復元にはPython 3.12以降（またはdata_filter搭載版）が必要です。\',\n    \'environment.busy\': \'環境が変更中です。インストール完了後に保存を再実行してください。\',\n    \'environment.unpinned_lock\': \'依存関係一覧に固定バージョン以外の行（editable・ローカルパス・pipオプション）があります。通常のpipパッケージへ置き換えてください。\',\n    \'cloudflared.download_failed\': \'cloudflaredのダウンロードに失敗しました。\',\n    \'cloudflared.sha256_mismatch\': \'cloudflaredのSHA256が公開値と一致しません。実行を中止します。\',\n    \'setup.not_colab\': \'このセットアップはroot権限のGoogle Colab Linux GPUランタイム専用です。\',\n    \'setup.already_done\': \'このランタイムはセットアップ済みです。再セットアップにはランタイムを削除して接続し直してください。\',\n    \'setup.invalid_ref\': \'ComfyUIのrefが不正です。\',\n    \'setup.interval_too_short\': \'保存間隔は60秒以上にしてください。\',\n    \'setup.invalid_apt_package\': \'APTパッケージ名が不正です。\',\n    \'setup.no_gpu\': \'Colabのランタイム設定でGPUを選択してください。\',\n    \'setup.fresh_environment\': \'保存済みの環境を復元せず、新しい環境を作成します。Drive上の保存は次回の保存から新しい世代に置き換わります。\',\n    \'setup.invalid_archive_name\': \'保存情報のアーカイブ名が不正です。\',\n    \'setup.python_changed\': \'保存時とColabのPythonバージョンが違います。環境移行が必要です。既存保存を上書きしません。\',\n    \'setup.archive_sha256_mismatch\': \'保存アーカイブのSHA256が一致しません。復元を中止します。\',\n    \'setup.restored\': \'Drive上の環境を復元しました。保存時刻:\',\n    \'setup.restored_nodes\': \'復元したカスタムノード:\',\n    \'setup.source_incomplete\': \'復元したComfyUIソースが不完全です。\',\n    \'setup.saved_apt_packages\': \'保存時に使われていた追加のOSパッケージ:\',\n    \'setup.saved_apt_packages_hint\': \'必要な場合は、1のAPT_PACKAGESへ追加して新しいランタイムで実行し直してください（自動では導入しません）。\',\n    \'setup.no_torch\': \'Colabの標準PyTorch CUDA環境を検出できません。GPUランタイムを確認してください。\',\n    \'setup.invalid_torch_mirror\': \'保存されたPyTorchミラーが不正です。\',\n    \'setup.copying_models\': \'DriveのモデルをColabへコピーします（モデル全体のローカル空き容量が必要です）。\',\n    \'setup.done\': \'セットアップ完了。Driveの保存先: MyDrive/\',\n    \'download.hf_repository\': \'Hugging FaceのリポジトリIDを author/repository 形式で指定してください。\',\n    \'download.hf_revision\': \'revisionを指定してください。\',\n    \'download.civitai_version\': \'CivitaiのモデルバージョンIDを正の整数で指定してください。\',\n    \'download.unknown_source\': \'ダウンロード元をHugging FaceまたはCivitaiから選択してください。\',\n    \'download.hf_url\': \'Hugging Faceは、Files and versionsで開いたファイルのURL（…/blob/… または …/resolve/…）を指定してください。\',\n    \'download.civitai_url\': \'Civitaiは、バージョンを選んだページのURL（…?modelVersionId=…）かダウンロードリンクを指定してください。\',\n    \'download.unsupported_url\': \'https://huggingface.co/ または https://civitai.com/ のURLを指定してください。\',\n    \'download.filename_has_folder\': \'保存名にフォルダーを含めないでください。保存先はMODEL_FOLDERで指定します。\',\n    \'download.not_https\': \'ダウンロード先には認証情報を含まないHTTPS URLが必要です。\',\n    \'download.local_redirect\': \'ローカルサービスへのリダイレクトを拒否しました。\',\n    \'download.unresolved_host\': \'ダウンロード先のホスト名を解決できません。\',\n    \'download.private_redirect\': \'プライベートIPへのリダイレクトを拒否しました。\',\n    \'download.redirect_without_target\': \'ダウンロードの転送先がありません。\',\n    \'download.http_error\': \'モデル取得失敗（HTTP {status}）。利用条件・アクセス権・トークンを確認してください。\',\n    \'download.not_a_model\': \'モデルではなくHTML/JSONが返されました。ファイル指定を確認してください。\',\n    \'download.empty\': \'ダウンロードファイルが空です。\',\n    \'download.too_many_redirects\': \'リダイレクト回数が上限に達しました。\',\n    \'download.host_not_allowed\': \'ダウンロード元はhuggingface.coまたはcivitai.comのみです。\',\n    \'download.url_has_query\': \'元URLにクエリや接続鍵を含めないでください。入力フォームからURLを生成してください。\',\n    \'download.invalid_sha256\': \'SHA256は64桁の16進数で指定してください。\',\n    \'download.exists\': \'Driveに同名モデルが存在します。別の保存名を指定してください。\',\n    \'download.sha256_mismatch\': \'モデルのSHA256が一致しません。確定ファイルとして保存しません。\',\n    \'download.no_filename\': \'配布元からファイル名を取得できませんでした。保存名を指定してください。\',\n    \'download.exists_not_overwritten\': \'Driveに同名モデルが存在します。上書きしません。別の保存名を指定してください。\',\n    \'download.saved\': \'Driveへ保存し、Colabへ反映しました:\',\n    \'download.connection_failed\': \'モデルのダウンロード通信に失敗しました。接続・アクセス権を確認してください。\',\n    \'proxy.unauthorized\': \'認証が必要です。ノートブックが表示したURLをブラウザーで開き、ペアリングコードを入力してください。\',\n    \'proxy.cross_origin\': \'異なるOriginからのアクセスは許可されていません。\',\n    \'proxy.websocket_error\': \'WebSocket中継エラー:\',\n    \'proxy.upstream_down\': \'ComfyUIが起動中、または停止しています。\',\n    \'proxy.origin_rejected\': \'別のサイトからの要求を拒否しました: Origin {origin} / Host {host}\',\n    \'proxy.access_required\': \'Cloudflare Accessの認証を確認できません。公開ホスト名のURLから接続してください。ログイン後もこの表示が出る場合は、シークレット CF_ACCESS_AUD のAUDタグを確認してください。\',\n    \'diagnosis.route\': \'  [{mark}] {status} {seconds:6.2f}秒 {size:>9}B {route}\',\n    \'diagnosis.assets\': \'  [{mark}] 画面のJS/CSS {total}件: 失敗{failed}件 / 展開後{megabytes:.1f}MB / {elapsed:.2f}秒 / 圧縮あり{compressed}件\',\n    \'diagnosis.failed_asset\': \'       {status} {seconds:.2f}秒 {route}\',\n    \'diagnosis.websocket\': \'  [{mark}] WebSocket 最初のメッセージ: {kind}\',\n    \'diagnosis.not_running\': \'ComfyUIが起動していません。6-1〜6-3のいずれかを実行してから診断してください。\',\n    \'diagnosis.target_comfyui\': \'ComfyUI本体（127.0.0.1:8188）\',\n    \'diagnosis.target_proxy\': \'認証プロキシ（127.0.0.1:8189）\',\n    \'diagnosis.target_public\': \'公開URL（Cloudflare経由）\',\n    \'diagnosis.failed_at\': \'結果: {label} で失敗しています。ここより手前は正常です。\',\n    \'diagnosis.all_ok\': \'結果: Colab側は公開URLまで正常です。画面が出ない場合はPC側（ブラウザー拡張・広告ブロック・セキュリティソフト）を確認してください。\',\n    \'diagnosis.colab_ok\': \'結果: ComfyUI本体と認証プロキシは正常です。Colabのポート転送はここからは確認できません。画面が出ない場合は、ブラウザーがこのノートブックと同じGoogleアカウントでログインしているか、サードパーティCookieの制限やブラウザー拡張が妨げていないかを確認してください。\',\n    \'diagnosis.access_ok\': \'結果: ComfyUI本体は正常です。認証プロキシとCloudflare側は、Accessのログインが必要なためここからは確認できません。画面が出ない場合は、7のtunnel.logと、Cloudflareのトンネル・Accessアプリケーションの設定を確認してください。\',\n    \'isolation.heading\': \'アプリ用ユーザー（{user}）から到達できるローカルサービス:\',\n    \'isolation.comfyui\': \'ComfyUI本体\',\n    \'isolation.proxy\': \'認証プロキシ\',\n    \'isolation.unix_sockets\': \'接続できたUnixソケット（待受{listening}件中{reachable}件）:\',\n    \'isolation.kernel\': \'カーネル:\',\n    \'isolation.root_capabilities\': \'rootの権限:\',\n    \'isolation.no_command\': \'コマンドなし\',\n    \'isolation.available\': \'利用可\',\n    \'isolation.unavailable\': \'利用不可\',\n    \'supervisor.autosave_failed\': \'環境の自動保存に失敗。終了前に保存セルを実行してください。\',\n    \'supervisor.save_before_restart_failed\': \'再起動前の保存失敗\',\n    \'supervisor.restarting\': \'ComfyUIの終了を検出。10秒後に再起動します。\',\n    \'supervisor.restart_limit\': \'ComfyUIの再起動上限に達しました。comfy.logを確認してください。\',\n    \'start.already_running\': \'既に起動済みです。再起動は停止セルの後に実行してください。\',\n    \'start.proxy_exited_port\': \'認証プロキシが終了しました。8189番ポートが使用中の可能性があります。ログセルのproxy.logを確認してください。\',\n    \'start.proxy_not_started\': \'認証プロキシが起動しませんでした。\',\n    \'start.comfyui_timeout\': \'ComfyUIが5分以内に起動しませんでした。ログセルを確認してください。\',\n    \'start.proxy_exited\': \'認証プロキシが終了しました。ログセルのproxy.logを確認してください。\',\n    \'start.url_auto_download\': \'Remote Connectionでは Auto-download outputs を有効にし、PC上の保存先を設定してください。\',\n    \'start.tunnel_exited\': \'Quick Tunnelが終了しました。tunnel.logを確認してください。\',\n    \'start.no_tunnel_url\': \'Quick Tunnel URLを取得できませんでした。\',\n    \'start.tunnel_heading\': \'Comfy Desktop > Remote Connection に入力するURL（通常のブラウザーでも開けます）:\',\n    \'start.pairing_code\': \'ペアリングコード: {code}（{minutes}分間・1回だけ有効）\',\n    \'start.pairing_note\': \'URLを開くとコードの入力画面が出ます。別の端末を追加するときは、6-2の NEW_PAIRING_CODE をONにして実行してください。停止するとURLと接続済みの端末は無効になります。\',\n    \'start.colab_heading\': \'このブラウザーで開くURL（このノートブックを実行しているGoogleアカウントでだけ開けます）:\',\n    \'start.colab_note\': \'Comfy Desktopや別の端末からは接続できません。生成物はComfyUIの画面から保存するか、9で回収してください。\',\n    \'start.colab_unavailable\': \'Colabのポート転送のアドレスを取得できませんでした。Colabのノートブック上で実行してください。\',\n    \'stop.saved\': \'Driveへの環境保存が完了しました。\',\n    \'stop.done\': \'停止しました。生成ファイルはランタイム削除前にPCで確認してください。\',\n    \'update.running\': \'ComfyUIが起動中です。10で停止（FLUSH_AND_UNMOUNTはOFF）してから実行してください。\',\n    \'update.unknown_ref\': \'ComfyUIに {ref} というバージョンが見つかりません。\',\n    \'update.already\': \'ComfyUIは既に {version} です。\',\n    \'update.starting\': \'ComfyUIを {previous} から {ref} へ更新します。\',\n    \'update.undone\': \'更新に失敗したため、元のバージョン（{version}）へ戻しました。Driveの保存内容は変わっていません。\',\n    \'update.undo_failed\': \'更新に失敗し、元のバージョンへも戻せませんでした。11をもう一度実行してください。\',\n    \'update.incomplete\': \'ComfyUIの更新が途中で止まっています。11をもう一度実行するか、ランタイムを削除してください（Driveには更新前の環境が残っています）。\',\n    \'update.resuming\': \'途中で止まった更新を元に戻します。\',\n    \'update.done\': \'ComfyUIを {previous} から {version} へ更新し、Driveへ保存しました。6-1〜6-3のいずれかを実行して起動してください。\',\n    \'pairing.title\': \'ComfyUIへ接続\',\n    \'pairing.prompt\': \'ノートブックに表示されたペアリングコードを入力してください。\',\n    \'pairing.button\': \'接続\',\n    \'pairing.wrong\': \'コードが違います。あと{remaining}回入力できます。\',\n    \'pairing.closed\': \'このコードは使えません。ノートブックの6-2で NEW_PAIRING_CODE をONにして実行し、新しいコードを発行してください。\',\n    \'pairing.not_running\': \'6-2で起動してから実行してください。\',\n    \'access.invalid_hostname\': \'公開ホスト名が不正です。comfy.example.com の形式で指定してください。\',\n    \'access.invalid_audience\': \'AUDタグが不正です。Colabのシークレット CF_ACCESS_AUD に、AccessアプリケーションのApplication Audience (AUD) Tag（64桁の英数字）を登録してください。\',\n    \'access.invalid_token\': \'トンネルのトークンが不正です。Colabのシークレット CF_TUNNEL_TOKEN に、eyJ で始まるトークンだけを登録してください。\',\n    \'access.tunnel_exited\': \'名前付きトンネルが終了しました。トークンが正しいか、7のtunnel.logを確認してください。\',\n    \'access.unprotected\': \'{hostname} でCloudflare Accessのログイン画面への転送を確認できませんでした（応答: {status}）。Accessアプリケーションがこのホスト名を保護しているか、DNSの設定が反映されているかを確認してください。設定の直後は、数分待ってから実行し直してください。起動していません。\',\n    \'access.heading\': \'Comfy Desktop > Remote Connection に入力するURL（通常のブラウザーでも開けます）:\',\n    \'access.note\': \'開くとCloudflare Accessのログイン画面が出ます。許可したアカウントでログインした端末だけが接続できます。停止するとこのURLは応答しなくなります。\',\n    \'access.team\': \'このホスト名を保護しているAccessのチームドメイン: {team}\',\n    \'access.wrong_target\': \'トンネルの公開ホスト名の転送先が http://127.0.0.1:8189 だけになっていません（現在の設定: {services}）。Cloudflareのトンネルの設定で、サービスの種類をHTTP、URLを 127.0.0.1:8189 にしてから実行し直してください。接続を中止しました。\',\n    \'access.target_unknown\': \'注意: トンネルの転送先の設定を読み取れませんでした。Cloudflareのトンネルの設定で、公開ホスト名の転送先が http://127.0.0.1:8189 になっていることを確認してください。\',\n    \'menu.stop\': \'停止\',\n    \'menu.title\': \'ComfyUIを停止\',\n    \'menu.explain\': \'環境をDriveへ保存してから接続を解除します。生成物はDriveへ保存されません。ランタイムを削除すると、PCへ取得していない出力は失われます。\',\n    \'menu.delete\': \'接続を解除してランタイムを削除\',\n    \'menu.disconnect\': \'接続を解除\',\n    \'menu.cancel\': \'キャンセル\',\n    \'menu.working\': \'保存して停止しています。完了するとこのページは切断されます。\',\n    \'menu.stopped\': \'接続を解除しました。再開するときは、ノートブックの6-1〜6-3のいずれかを実行してください。\',\n    \'menu.deleted\': \'接続を解除しました。ランタイムの削除には少し時間がかかります。Colabの画面でランタイムが切断されたことを確認してください。\',\n    \'menu.failed\': \'停止できませんでした。ノートブックの10で停止してください。\',\n    \'menu.slow\': \'時間がかかっています。進まない場合は、ノートブックの10で停止するかColabの画面からランタイムを削除してください。\',\n}\nif MESSAGES is None:\n    # Run from the source tree: the Japanese messages of translations.json next to this file.\n    MESSAGES = {f"{section}.{name}": entry["JP"] for section, entries in json.loads(\n        Path(__file__).with_name("translations.json").read_text(encoding="utf-8"))["runtime"].items()\n        for name, entry in entries.items()}\n\n\ndef t(key, **values):\n    """The message translations.json holds as runtime.<key>, with its values filled in."""\n    return MESSAGES[key].format(**values)\n\n\ndef excluded_path(parts):\n    return "__pycache__" in parts or ".venv" in parts or (\n        len(parts) >= 2 and parts[0] == "ComfyUI" and parts[1] in EXCLUDED)\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as file:\n        for chunk in iter(lambda: file.read(4 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef private_write(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True, mode=0o700)\n    path.write_text(data, encoding="utf-8")\n    path.chmod(0o600)\n\n\ndef safe_relative(value):\n    if "\\\\" in value or ":" in value or "\\x00" in value:\n        raise ValueError(t("path.use_slashes"))\n    p = PurePosixPath(value)\n    if not value or p.is_absolute() or any(x in {"..", "."} for x in value.split("/")):\n        raise ValueError(t("path.unsafe"))\n    return str(p)\n\n\nGPU_ENVIRONMENT_KEYS = (\n    "LD_LIBRARY_PATH", "LD_PRELOAD", "CUDA_VISIBLE_DEVICES", "CUDA_HOME", "CUDA_PATH",\n    "CUDA_MODULE_LOADING", "CUDA_DEVICE_ORDER", "CUDA_CACHE_PATH",\n    "NVIDIA_VISIBLE_DEVICES", "NVIDIA_DRIVER_CAPABILITIES", "PYTORCH_NVML_BASED_CUDA_CHECK",\n)\n\n\ndef application_environment():\n    """Preserve GPU loader/device settings without copying notebook credentials."""\n    env = {"PATH": f"{BASE / \'.venv/bin\'}:/usr/local/bin:/usr/bin:/bin",\n           "HOME": str(BASE / "home"), "LANG": "C.UTF-8",\n           "PYTHONUNBUFFERED": "1", "VIRTUAL_ENV": str(BASE / ".venv"),\n           "PIP_DISABLE_PIP_VERSION_CHECK": "1",\n           # Colab\'s global pip.conf logs to /var/log/pip.log, which only root can write:\n           # as the application user every log record would fail with a traceback.\n           "PIP_CONFIG_FILE": "/dev/null"}\n    for name in GPU_ENVIRONMENT_KEYS:\n        if name in os.environ:\n            env[name] = os.environ[name]\n    if "LD_LIBRARY_PATH" in env:\n        # An empty or relative entry means the current directory, which the application user controls.\n        env["LD_LIBRARY_PATH"] = ":".join(p for p in env["LD_LIBRARY_PATH"].split(":") if p.startswith("/"))\n    return env\n\n\ndef drops_privileges():\n    return sys.platform == "linux" and os.geteuid() == 0\n\n\ndef app_command(args):\n    """Run args as the unprivileged application user; setuid programs cannot give root back."""\n    args = list(map(str, args))\n    if not drops_privileges():\n        return args\n    # Absolute: the application\'s PATH starts in a directory that user can write, and root\n    # is still root while this program is looked up.\n    return [SETPRIV, "--reuid", APP_USER, "--regid", APP_USER, "--init-groups", "--no-new-privs", "--", *args]\n\n\ndef run(args, *, app=False, capture=False, cwd=None, timeout=None):\n    args = list(map(str, args))\n    env = application_environment() if app else os.environ.copy()\n    # Describe the stage without printing command arguments or credentials.\n    stage = Path(args[0]).name\n    if "-m" in args and args.index("-m") + 1 < len(args):\n        module_index = args.index("-m") + 1\n        stage += " -m " + args[module_index]\n        words = {"pip": 1, "uv": 2}.get(args[module_index], 0)\n        rest = args[module_index + 1:]\n        if rest[:1] == ["--python"]:  # pip aimed at another environment: name the subcommand, not the flag.\n            rest = rest[2:]\n        stage += "".join(" " + word for word in rest[:words])\n    if app:\n        args = app_command(args)\n    if not capture:\n        print(t("command.running") + stage, flush=True)\n    started = time.monotonic()\n    # Child-process file descriptors can bypass Colab\'s notebook output capture.\n    # Always collect output and explicitly display it through notebook Python.\n    if app and drops_privileges():\n        from types import SimpleNamespace\n        try:\n            code, out, err = bounded(args, env=env, cwd=cwd, limit=64 * 1024 ** 2, timeout=timeout or 2 * APP_TIMEOUT)\n        except RuntimeError as failure:\n            raise RuntimeError(t("command.failed", stage=stage, failure=failure)) from None\n        result = SimpleNamespace(returncode=code, stdout=out.decode(errors="replace"), stderr=err.decode(errors="replace"))\n    else:\n        result = subprocess.run(args, env=env, cwd=cwd, capture_output=True, text=True, timeout=timeout)\n    if result.returncode or not capture:\n        if result.stdout:\n            print(result.stdout, end="" if result.stdout.endswith("\\n") else "\\n", flush=True)\n        if result.stderr:\n            print(result.stderr, end="" if result.stderr.endswith("\\n") else "\\n", flush=True)\n    if result.returncode:\n        raise RuntimeError(t("command.exit_code", stage=stage, code=result.returncode))\n    if not capture:\n        print(t("command.done", stage=stage, seconds=time.monotonic() - started), flush=True)\n    return result.stdout if capture else None\n\n\ndef bounded(command, *, env, limit, timeout, stdin=None, sink=None, cwd=None):\n    """Run a command the application user controls and collect what it prints.\n\n    It cannot make root wait past timeout or take more than limit bytes, whether it stalls,\n    floods its output, or leaves a child of its own holding the pipes open.\n    Returns (exit code, output unless sink was given, end of the error output)."""\n    import select\n    with tempfile.TemporaryFile() as spool:\n        if isinstance(stdin, bytes):\n            spool.write(stdin)\n            spool.seek(0)\n            stdin = spool\n        process = subprocess.Popen(command, env=env, cwd=cwd, stdin=stdin or subprocess.DEVNULL,\n                                   stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n    deadline = time.monotonic() + timeout\n    streams = {process.stdout.fileno(): True, process.stderr.fileno(): False}\n    output, errors, total, failure = [], b"", 0, None\n    try:\n        while streams and not failure:\n            remaining = deadline - time.monotonic()\n            if remaining <= 0:\n                failure = t("command.timed_out")\n                break\n            ready, _, _ = select.select(list(streams), [], [], min(remaining, 0.5))\n            for fd in ready:\n                chunk = os.read(fd, 1024 * 1024)\n                if not chunk:\n                    del streams[fd]\n                elif not streams[fd]:\n                    errors = (errors + chunk)[-1024 * 1024:]\n                elif total + len(chunk) > limit:\n                    failure = t("command.output_limit")\n                    break\n                else:\n                    total += len(chunk)\n                    sink.write(chunk) if sink else output.append(chunk)\n            if not ready and process.poll() is not None:\n                break  # It exited; anything still holding the pipes is not waited for.\n        if not failure:\n            try:\n                process.wait(timeout=max(deadline - time.monotonic(), 0))\n            except subprocess.TimeoutExpired:\n                failure = t("command.timed_out")\n    finally:\n        if process.poll() is None:\n            process.kill()\n        process.stdout.close()\n        process.stderr.close()\n        process.wait()\n    if failure:\n        raise RuntimeError(failure)\n    return process.returncode, b"".join(output), errors\n\n\ndef stop_app_processes():\n    """End everything the application user still runs, including what a custom node detached\n    from ComfyUI so that it would outlive a stop."""\n    if not drops_privileges():\n        return\n    import pwd\n    try:\n        uid = pwd.getpwnam(APP_USER).pw_uid\n    except KeyError:\n        return\n    # As that user, signal every process it may signal: all of its own, in one step.\n    subprocess.run(app_command(["/usr/bin/kill", "-KILL", "-1"]), env={}, capture_output=True)\n    for _ in range(50):\n        alive = False\n        for entry in Path("/proc").iterdir():\n            try:\n                if entry.name.isdigit() and entry.stat().st_uid == uid:\n                    alive = True\n                    os.kill(int(entry.name), signal.SIGKILL)\n            except (FileNotFoundError, ProcessLookupError):\n                pass\n        if not alive:\n            return\n        time.sleep(0.1)\n    raise RuntimeError(t("user.processes_remain"))\n\n\ndef ensure_app_user():\n    """Create the account ComfyUI, pip and custom nodes run as. It cannot enter PRIVATE (Drive, keys)."""\n    if not Path(SETPRIV).is_file():\n        raise RuntimeError(t("user.setpriv_missing", setpriv=SETPRIV))\n    if subprocess.run(["id", "-u", APP_USER], capture_output=True).returncode:\n        run(["useradd", "--system", "--user-group", "--no-create-home", "--home-dir", BASE / "home",\n             "--shell", "/usr/sbin/nologin", APP_USER])\n    # GPU device nodes are normally open to everyone; join their groups where they are not.\n    for device in [*Path("/dev").glob("nvidia*"), *Path("/dev/dri").glob("*")]:\n        info = device.stat()\n        if stat.S_ISCHR(info.st_mode) and info.st_mode & 0o060 and not info.st_mode & 0o006 and info.st_gid:\n            if subprocess.run(["getent", "group", str(info.st_gid)], capture_output=True).returncode:\n                run(["groupadd", "--gid", info.st_gid, f"{APP_USER}-device-{info.st_gid}"])\n            run(["usermod", "--append", "--groups", info.st_gid, APP_USER])\n\n\ndef create_app_tree():\n    """Make the application user\'s working directories. Root hands over only what it has just\n    made and never follows a link: on a rerun after a failed setup, the tree may already hold\n    whatever that user\'s code left there."""\n    import pwd\n    account = pwd.getpwnam(APP_USER)\n    created = []\n    for directory in (BASE, *(BASE / name for name in ("environment", "models", "output", "temp", "input", "home", "database"))):\n        try:\n            os.mkdir(directory)\n        except FileExistsError:\n            continue\n        created.append(directory)\n    for directory in reversed(created):  # BASE last, so nothing can be swapped underneath meanwhile.\n        os.chown(directory, account.pw_uid, account.pw_gid, follow_symlinks=False)\n\n\ndef app_tree_action(action, args, source, sink):\n    """File operations inside BASE. They run as the application user, so a link planted there\n    by a custom node cannot redirect root to Drive, the keys or system files."""\n    envdir = BASE / "environment"\n    repo = envdir / "ComfyUI"\n    if action == "put":\n        target = contained_path(BASE, args[0])\n        target.parent.mkdir(parents=True, exist_ok=True)\n        fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".part", dir=target.parent)\n        try:\n            with os.fdopen(fd, "wb") as file:\n                shutil.copyfileobj(source, file, 4 * 1024 * 1024)\n            if len(args) > 1:\n                os.utime(temporary, ns=(int(args[1]), int(args[1])))\n            os.replace(temporary, target)\n        finally:\n            Path(temporary).unlink(missing_ok=True)\n    elif action == "get":\n        with contained_path(BASE, args[0]).open("rb") as file:\n            shutil.copyfileobj(file, sink, 4 * 1024 * 1024)\n    elif action == "prepare":\n        # Keep the core default and Manager\'s direct models path on the same storage.\n        for name in MODEL_FOLDERS:\n            (BASE / "models" / name).mkdir(parents=True, exist_ok=True)\n        repo_models = repo / "models"\n        if repo_models.exists() and not repo_models.is_symlink():\n            shutil.copytree(repo_models, BASE / "models", dirs_exist_ok=True)\n            shutil.rmtree(repo_models)\n        if not repo_models.exists():\n            repo_models.symlink_to(BASE / "models", target_is_directory=True)\n        model_cfg = {"base_path": str(BASE / "models"), "is_default": True, **{name: name for name in MODEL_FOLDERS}}\n        (BASE / "models.yaml").write_text(json.dumps({"colab_drive_cache": model_cfg}))  # JSON is valid YAML.\n        for name in ("user", "custom_nodes"):\n            (repo / name).mkdir(exist_ok=True)\n        if (repo / "user/comfyui.db").exists():\n            shutil.copyfile(repo / "user/comfyui.db", BASE / "database/comfyui.db")\n    elif action == "restore":\n        fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".tar.gz", dir=BASE)\n        try:\n            with os.fdopen(fd, "wb") as file:\n                shutil.copyfileobj(source, file, 4 * 1024 * 1024)\n            restore_archive(temporary, envdir)\n        finally:\n            Path(temporary).unlink(missing_ok=True)\n    elif action == "backup-database":\n        # SQLite\'s online backup includes committed WAL changes consistently.\n        import sqlite3\n        with sqlite3.connect(f"file:{BASE / \'database/comfyui.db\'}?mode=ro", uri=True) as database:\n            with sqlite3.connect(repo / "user/comfyui.db") as target:\n                database.backup(target)\n    elif action == "archive":\n        with tarfile.open(fileobj=sink, mode="w|gz", compresslevel=1) as tf:\n            for child in sorted(envdir.iterdir()):\n                tf.add(child, arcname=child.name, filter=archive_filter)\n    elif action == "zip-output":\n        import zipfile\n        root = BASE / "output"\n        count = 0\n        with zipfile.ZipFile(sink, "w", compression=zipfile.ZIP_STORED) as archive:\n            for parent, dirs, files in os.walk(root, followlinks=False):\n                dirs[:] = [d for d in dirs if not (Path(parent) / d).is_symlink()]\n                for name in files:\n                    file = Path(parent) / name\n                    if file.is_symlink() or not file.is_file():\n                        continue\n                    archive.write(file, file.relative_to(root))\n                    count += 1\n        if not count:\n            raise RuntimeError(t("app.no_outputs"))\n    else:\n        raise ValueError(action)\n\n\ndef app_action(action, *args, stdin=None, stdout=None, limit=APP_OUTPUT_LIMIT, timeout=APP_TIMEOUT):\n    """Perform app_tree_action as the application user. stdin is bytes or a file; stdout is a file."""\n    args = list(map(str, args))\n    if not drops_privileges():\n        import io\n        sink = stdout or io.BytesIO()\n        app_tree_action(action, args, io.BytesIO(stdin) if isinstance(stdin, bytes) else stdin, sink)\n        return b"" if stdout else sink.getvalue()\n    command = app_command([sys.executable, Path(__file__).resolve(), "app", BASE, action, *args])\n    try:\n        code, output, errors = bounded(command, env=application_environment(), stdin=stdin, sink=stdout,\n                                       limit=limit, timeout=timeout)\n    except RuntimeError as failure:\n        raise RuntimeError(t("app.failed", action=action, failure=failure)) from None\n    if code:\n        detail = errors.decode(errors="replace").strip().splitlines()[-1:]\n        raise RuntimeError(t("app.failed_detail", action=action) + "".join(detail))\n    return output\n\n\ndef receive_file(target, relative, times=None, limit=APP_OUTPUT_LIMIT):\n    """Copy a file out of the application\'s tree to a root-only destination, atomically."""\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".part", dir=target.parent)\n    try:\n        with os.fdopen(fd, "wb") as file:\n            app_action("get", relative, stdout=file, limit=limit)\n        if times:\n            os.utime(temporary, ns=times)\n        os.replace(temporary, target)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\ndef installed_apt_packages():\n    """Names dpkg has installed, including names they provide: libglib2.0-0 is provided by\n    libglib2.0-0t64 on current Ubuntu."""\n    try:\n        listing = subprocess.run(["dpkg-query", "-W", "-f", "${db:Status-Abbrev}\\t${Package}\\t${Provides}\\n"],\n                                 capture_output=True, text=True).stdout\n    except OSError:\n        return set()\n    names = set()\n    for line in listing.splitlines():\n        status, _, rest = line.partition("\\t")\n        package, _, provides = rest.partition("\\t")\n        if status.startswith("ii"):\n            names.add(package)\n            names.update(item.split("(")[0].strip() for item in provides.split(",") if item.strip())\n    return names\n\n\nAPT_INDEX_REFRESHED = []\n\n\ndef apt_install(packages):\n    """Install the packages that are missing. Colab ships most of them, and refreshing the\n    package index alone takes longer than everything else here."""\n    missing = sorted(set(packages) - installed_apt_packages())\n    if not missing:\n        return\n    if not APT_INDEX_REFRESHED:\n        run(["apt-get", "update", "-qq"])\n        APT_INDEX_REFRESHED.append(True)\n    run(["apt-get", "install", "-y", "-qq", *missing])\n\n\ndef install_packages(*args):\n    """Install into the application\'s virtual environment with uv. It takes the same indexes\n    and pins as pip but downloads and unpacks the multi-gigabyte PyTorch stack in parallel."""\n    py = BASE / ".venv/bin/python"\n    run([py, "-m", "uv", "pip", "install", "--python", py, *args], app=True)\n\n\ndef create_venv():\n    """Create the application\'s virtual environment with pip and uv in it.\n\n    The notebook\'s own pip installs them, so nothing is needed from apt: installing\n    pythonX.Y-venv there upgrades the interpreter the running kernel was started from."""\n    venv = BASE / ".venv"\n    py = venv / "bin/python"\n    run([sys.executable, "-m", "venv", "--without-pip", venv], app=True, capture=True)\n    try:\n        run([sys.executable, "-m", "pip", "--python", py, "install", "pip", "uv"], app=True)\n    except RuntimeError:\n        print(t("venv.pip_fallback"), flush=True)\n        ensure_venv_support()\n        run([sys.executable, "-m", "venv", venv], app=True, capture=True)\n        run([py, "-m", "pip", "install", "uv"], app=True)\n    return py\n\n\ndef ensure_venv_support():\n    """Install ensurepip for the notebook\'s Python version when it is absent."""\n    import importlib.util\n    if importlib.util.find_spec("ensurepip") is not None:\n        return\n    version = f"{sys.version_info.major}.{sys.version_info.minor}"\n    package = f"python{version}-venv"\n    print(t("venv.ensurepip_missing", version=version, package=package), flush=True)\n    try:\n        apt_install([package])\n    except RuntimeError:\n        raise RuntimeError(t("venv.package_failed", package=package)) from None\n    importlib.invalidate_caches()\n    if importlib.util.find_spec("ensurepip") is None:\n        raise RuntimeError(t("venv.ensurepip_unavailable", package=package, python=sys.executable))\n\n\ndef mount_drive():\n    """Mount Drive under a root-only parent, out of reach of the application user."""\n    if sys.platform != "linux" or os.geteuid() != 0:\n        raise RuntimeError(t("drive.needs_root"))\n    if PRIVATE.is_symlink() or DRIVE_MOUNT.is_symlink():\n        raise RuntimeError(t("drive.private_symlink"))\n    PRIVATE.mkdir(mode=0o700, exist_ok=True)\n    PRIVATE.chmod(0o700)\n    if PRIVATE.stat().st_uid != 0:\n        raise RuntimeError(t("drive.private_owner"))\n    for line in Path("/proc/mounts").read_text().splitlines():\n        fields = line.split()\n        if len(fields) > 2 and fields[2] == "fuse.drive" and fields[1] != str(DRIVE_MOUNT):\n            raise RuntimeError(t("drive.mounted_elsewhere"))\n    from google.colab import drive\n    drive.mount(str(DRIVE_MOUNT))\n    PRIVATE.chmod(0o700)\n    if not (DRIVE_MOUNT / "MyDrive").is_dir():\n        raise RuntimeError(t("drive.no_mydrive"))\n\n\ndef contained_path(root, relative):\n    root = Path(root)\n    rel = safe_relative(relative)\n    target = root / rel\n    if root.is_symlink() or not target.resolve().is_relative_to(root.resolve()):\n        raise ValueError(t("path.outside_folder"))\n    current = root\n    for part in PurePosixPath(rel).parts:\n        current /= part\n        if current.is_symlink():\n            raise ValueError(t("path.symlink"))\n    return target\n\n\ndef drive_root(config):\n    mydrive = DRIVE_MOUNT / "MyDrive"\n    if not mydrive.is_dir():\n        raise RuntimeError(t("drive.not_mounted"))\n    return contained_path(mydrive, config["drive_folder"])\n\n\ndef atomic_copy(source, destination):\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    fd, temporary = tempfile.mkstemp(prefix=".comfy-", suffix=".part", dir=destination.parent)\n    os.close(fd)\n    try:\n        shutil.copy2(source, temporary)\n        os.replace(temporary, destination)\n    finally:\n        Path(temporary).unlink(missing_ok=True)\n\n\ndef regular_files(root):\n    for parent, dirs, files in os.walk(root, followlinks=False):\n        dirs[:] = [d for d in dirs if not (Path(parent) / d).is_symlink()]\n        for name in files:\n            file = Path(parent) / name\n            if file.is_symlink() or not file.is_file() or name.endswith((".part", ".tmp", ".download")):\n                continue\n            if name.startswith(".comfy-") or not file.stat().st_size:\n                continue  # Empty files are ComfyUI\'s "put_models_here" markers, not models.\n            yield file\n\n\ndef sync_models(config):\n    """Drive is authoritative; refresh the private runtime\'s working copy."""\n    source = drive_root(config) / "models"\n    source.mkdir(exist_ok=True)\n    copied = 0\n    origin_file = PRIVATE / "model-origins.json"\n    origins = set(json.loads(origin_file.read_text())) if origin_file.exists() else set()\n    for file in regular_files(source):\n        relative = safe_relative(file.relative_to(source).as_posix())\n        origins.add(relative)\n        target = BASE / "models" / relative\n        st = file.stat()\n        if target.exists():\n            old = target.stat()\n            if old.st_size == st.st_size and abs(old.st_mtime - st.st_mtime) < 1:\n                continue\n        with file.open("rb") as stream:\n            app_action("put", "models/" + relative, st.st_mtime_ns, stdin=stream)\n        copied += 1\n    private_write(origin_file, json.dumps(sorted(origins)))\n    print(t("models.synced", copied=copied))\n    return copied\n\n\ndef save_new_cache_models(config, force=False):\n    # Preserve Manager downloads without overwriting manually updated Drive files.\n    destination = drive_root(config) / "models"\n    destination.mkdir(exist_ok=True)\n    origin_file = PRIVATE / "model-origins.json"\n    origins = set(json.loads(origin_file.read_text())) if origin_file.exists() else set()\n    for file in regular_files(BASE / "models"):\n        relative = file.relative_to(BASE / "models").as_posix()\n        if relative in origins:\n            continue\n        st = file.stat()\n        if not force and time.time() - st.st_mtime < 30:\n            continue\n        target = contained_path(destination, relative)\n        if not target.exists():\n            # No more than the size root saw: the file is the application user\'s to change.\n            receive_file(target, "models/" + relative, (st.st_atime_ns, st.st_mtime_ns), limit=st.st_size)\n        origins.add(relative)\n    private_write(origin_file, json.dumps(sorted(origins)))\n\n\ndef archive_filter(member):\n    parts = PurePosixPath(member.name).parts\n    if excluded_path(parts) or any(p.endswith(".pyc") for p in parts):\n        return None\n    # Third-party symlinks could capture credentials or files outside the workspace.\n    if member.issym() or member.islnk() or not (member.isfile() or member.isdir()):\n        return None\n    return member\n\n\ndef restore_archive(archive, dest):\n    with tarfile.open(archive, "r:gz") as tf:\n        for member in tf.getmembers():\n            safe_relative(member.name)\n            if member.issym() or member.islnk() or not (member.isfile() or member.isdir()):\n                raise ValueError(t("environment.archive_special_files"))\n        if not hasattr(tarfile, "data_filter"):\n            raise RuntimeError(t("environment.archive_needs_data_filter"))\n        tf.extractall(dest, filter="data")\n\n\ndef fingerprint(root):\n    entries = []\n    for parent, dirs, files in os.walk(root, followlinks=False):\n        dirs[:] = sorted(d for d in dirs if not excluded_path((Path(parent) / d).relative_to(root).parts)\n                         and not (Path(parent) / d).is_symlink())\n        for name in sorted(files):\n            p = Path(parent) / name\n            if p.is_symlink() or name.endswith(".pyc"):\n                continue\n            try:\n                st = p.stat()\n                entries.append((str(p.relative_to(root)), st.st_size, st.st_mtime_ns))\n            except FileNotFoundError:\n                continue\n    return hashlib.sha256(json.dumps(entries).encode()).hexdigest()\n\n\ndef checkpoint(config, force=False):\n    import fcntl\n    with (PRIVATE / "checkpoint.lock").open("w") as lock:\n        fcntl.flock(lock, fcntl.LOCK_EX)\n        refuse_unfinished_update()\n        envdir = BASE / "environment"\n        save_new_cache_models(config, force=force)\n        storage = drive_root(config) / "environment"\n        storage.mkdir(exist_ok=True)\n        database = BASE / "database/comfyui.db"\n        if database.exists():\n            signature = [(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in\n                         (database, database.with_name(database.name + "-wal")) if p.exists()]\n            signature_text = json.dumps(signature)\n            db_marker = PRIVATE / "database-signature"\n            if force or not db_marker.exists() or db_marker.read_text() != signature_text:\n                app_action("backup-database")\n                db_marker.write_text(signature_text)\n        write_lock()\n        marker = PRIVATE / "last-fingerprint"\n        before = fingerprint(envdir)\n        if not force and marker.exists() and marker.read_text() == before:\n            return False\n        staging = PRIVATE / "environment.tar.gz"\n        with staging.open("wb") as file:\n            app_action("archive", stdout=file, limit=ARCHIVE_LIMIT, timeout=1800)\n        after = fingerprint(envdir)\n        if before != after:\n            staging.unlink(missing_ok=True)\n            raise RuntimeError(t("environment.busy"))\n        # Generation files + last-written pointer keep the preceding good snapshot.\n        digest = file_hash(staging)\n        name = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + digest[:12] + ".tar.gz"\n        atomic_copy(staging, contained_path(storage, name))\n        pointer = {"archive": name, "sha256": digest, "python": config["python"],\n                   "saved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n        private_write(PRIVATE / "latest.json", json.dumps(pointer))\n        atomic_copy(PRIVATE / "latest.json", contained_path(storage, "latest.json"))\n        marker.write_text(after)\n        private_write(PRIVATE / "checkpoint-status.json", json.dumps({"ok": True, **pointer}))\n        staging.unlink(missing_ok=True)\n        # Retain three completed generations; delete only our strictly named archives.\n        archives = [item for item in storage.iterdir()\n                    if not item.is_symlink() and re.fullmatch(r"\\d{8}T\\d{6}Z-[a-f0-9]{12}\\.tar\\.gz", item.name)]\n        archives.sort(key=lambda item: (item.stat().st_mtime_ns, item.name), reverse=True)\n        for old in archives[3:]:\n            if old.name != name:\n                contained_path(storage, old.name).unlink()\n        return True\n\n\ndef write_lock():\n    """Record the installed packages, to install the same ones on the next runtime."""\n    py = BASE / ".venv/bin/python"\n    # No editable or local paths: those cannot survive a new Colab VM.\n    packages = checked_lock(run([py, "-m", "pip", "freeze"], app=True, capture=True, timeout=300))\n    lock = "environment/requirements.lock.txt"\n    if not (BASE / lock).exists() or app_action("get", lock).decode() != packages:\n        app_action("put", lock, stdin=packages.encode())\n\n\nLOCK_LINE = re.compile(r"[A-Za-z0-9][A-Za-z0-9._-]*(==[A-Za-z0-9._+!-]+| @ (git\\+)?https://\\S+)")\n\n\ndef checked_lock(text):\n    """Accept only pinned packages: a lock is installed again on every new runtime, so pip\n    options, editable installs and local paths must not ride along in it."""\n    if any(line.strip() and not LOCK_LINE.fullmatch(line) for line in text.splitlines()):\n        raise RuntimeError(t("environment.unpinned_lock"))\n    return text\n\n\ndef checked_torch_index(url):\n    if not isinstance(url, str) or not re.fullmatch(r"https://download\\.pytorch\\.org/whl/cu\\d+", url):\n        raise RuntimeError(t("setup.invalid_torch_mirror"))\n    return url\n\n\ndef unlisted_apt_packages(saved, requested):\n    """Package names recorded with a saved environment that this setup was not asked for."""\n    names = saved.get("apt_packages", []) if isinstance(saved, dict) else []\n    return sorted({name for name in names if isinstance(name, str)\n                   and re.fullmatch(r"[a-z0-9][a-z0-9+.-]*", name)} - set(requested))\n\n\ndef install_cloudflared():\n    """Download the pinned release and refuse to run anything else."""\n    import requests\n    binary = PRIVATE / "cloudflared"\n    partial = PRIVATE / "cloudflared.part"\n    url = ("https://github.com/cloudflare/cloudflared/releases/download/"\n           + CLOUDFLARED_VERSION + "/cloudflared-linux-amd64")\n    try:\n        with requests.get(url, stream=True, timeout=60) as res:\n            if not res.ok:\n                raise RuntimeError(t("cloudflared.download_failed"))\n            with partial.open("wb") as f:\n                for chunk in res.iter_content(1024 * 1024):\n                    f.write(chunk)\n        if file_hash(partial) != CLOUDFLARED_SHA256:\n            raise RuntimeError(t("cloudflared.sha256_mismatch"))\n        partial.chmod(0o700)\n        os.replace(partial, binary)\n    finally:\n        partial.unlink(missing_ok=True)\n\n\ndef setup(*, root_name="ComfyUI-Colab", comfy_ref="v0.39.0",\n          checkpoint_seconds=120, apt_packages=(), restore=True):\n    if sys.platform != "linux" or not Path("/content").exists() or os.geteuid() != 0:\n        raise RuntimeError(t("setup.not_colab"))\n    if CONFIG.exists():\n        raise RuntimeError(t("setup.already_done"))\n    safe_relative(root_name)\n    if not re.fullmatch(r"[A-Za-z0-9._/-]+", comfy_ref) or comfy_ref.startswith("-"):\n        raise ValueError(t("setup.invalid_ref"))\n    if checkpoint_seconds < 60:\n        raise ValueError(t("setup.interval_too_short"))\n    if any(not re.fullmatch(r"[a-z0-9][a-z0-9+.-]*", p) for p in apt_packages):\n        raise ValueError(t("setup.invalid_apt_package"))\n    if subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL).returncode:\n        raise RuntimeError(t("setup.no_gpu"))\n    if PRIVATE.is_symlink():\n        raise RuntimeError(t("drive.private_symlink"))\n    PRIVATE.mkdir(mode=0o700, exist_ok=True)\n    PRIVATE.chmod(0o700)\n    apt_install(["git", "ffmpeg", "libgl1", "libglib2.0-0", *apt_packages])\n    mount_drive()\n    config = {"drive_folder": root_name,\n              "python": f"{sys.version_info.major}.{sys.version_info.minor}",\n              "checkpoint_seconds": checkpoint_seconds}\n    storage = drive_root(config)\n    for directory in (storage, storage / "environment", *(storage / "models" / name for name in MODEL_FOLDERS)):\n        directory.mkdir(parents=True, exist_ok=True)\n    # Test the selected folder, not any other location in Drive.\n    probe = storage / "environment/write-probe.txt"\n    probe.write_text("ComfyUI-Colab write test")\n    # The application user owns its working tree from here on. Root reads and writes inside\n    # it only through app_action, because code running there can plant links.\n    envdir = BASE / "environment"\n    ensure_app_user()\n    create_app_tree()\n    pointer_path = storage / "environment/latest.json"\n    if pointer_path.exists() and not restore:\n        print(t("setup.fresh_environment"))\n    elif pointer_path.exists():\n        pointer = json.loads(pointer_path.read_text())\n        archive_name = safe_relative(pointer["archive"])\n        if "/" in archive_name or not archive_name.endswith(".tar.gz"):\n            raise RuntimeError(t("setup.invalid_archive_name"))\n        if pointer["python"] != config["python"]:\n            raise RuntimeError(t("setup.python_changed"))\n        archive = PRIVATE / "restore.tar.gz"\n        atomic_copy(contained_path(storage / "environment", archive_name), archive)\n        if file_hash(archive) != pointer["sha256"]:\n            raise RuntimeError(t("setup.archive_sha256_mismatch"))\n        with archive.open("rb") as stream:\n            app_action("restore", stdin=stream, timeout=2 * APP_TIMEOUT)\n        archive.unlink()\n        print(t("setup.restored"), pointer["saved_at"])\n        nodes = sorted(item.name for item in (envdir / "ComfyUI/custom_nodes").glob("*")\n                       if item.is_dir() and item.name != "__pycache__")\n        # Restored nodes run again on this runtime; show them so an unwanted one is noticed.\n        print(t("setup.restored_nodes"), ", ".join(nodes) if nodes else t("common.none"))\n    repo = envdir / "ComfyUI"\n    if not repo.exists():\n        run(["git", "clone", "https://github.com/Comfy-Org/ComfyUI.git", repo], app=True)\n        run(["git", "checkout", comfy_ref], app=True, cwd=repo)\n    if not (repo / "main.py").exists():\n        raise RuntimeError(t("setup.source_incomplete"))\n    app_action("prepare")\n    previous_meta = (json.loads(app_action("get", "environment/runtime.json"))\n                     if (envdir / "runtime.json").exists() else {})\n    # The saved list is a file the application user could rewrite, and apt runs as root:\n    # name what was used before and leave the decision to install it to the notebook\'s setting.\n    wanted = unlisted_apt_packages(previous_meta, apt_packages)\n    if wanted:\n        print(t("setup.saved_apt_packages"), " ".join(wanted))\n        print(t("setup.saved_apt_packages_hint"))\n    # Match the CUDA build already supported by the current Colab runtime.\n    # The mirror is persisted because pip freeze can contain +cuXXX versions.\n    lock = envdir / "requirements.lock.txt"\n    locked = lock.exists()\n    if locked:\n        checked_lock(app_action("get", "environment/requirements.lock.txt").decode())\n        torch_index = previous_meta.get("torch_index_url", "")\n        torch_packages = []\n    else:\n        import importlib.metadata\n        import torch\n        cuda = torch.version.cuda\n        if not cuda or not re.fullmatch(r"\\d+\\.\\d+", cuda):\n            raise RuntimeError(t("setup.no_torch"))\n        torch_index = "https://download.pytorch.org/whl/cu" + cuda.replace(".", "")\n        torch_packages = ["torch==" + importlib.metadata.version("torch"),\n                          "torchvision==" + importlib.metadata.version("torchvision")]\n    checked_torch_index(torch_index)\n    app_action("put", "environment/runtime.json", stdin=json.dumps(\n        {"apt_packages": sorted(apt_packages), "python": config["python"], "torch_index_url": torch_index}).encode())\n    py = create_venv()\n    if locked:\n        # pip looks for each package on every index; uv does only when told to.\n        install_packages("--index-strategy", "unsafe-best-match", "--extra-index-url", torch_index, "-r", lock)\n    else:\n        install_packages("--index-url", torch_index, *torch_packages)\n        install_packages("-r", repo / "requirements.txt", "-r", repo / "manager_requirements.txt")\n        # Explicit requirements are useful for custom nodes added outside Manager.\n        for req in sorted((repo / "custom_nodes").glob("*/requirements.txt")):\n            install_packages("-r", req)\n    run([py, "-m", "pip", "check"], app=True)\n    run([py, "-c", "import torch; print(\'PyTorch:\', torch.__version__); print(\'CUDA build:\', torch.version.cuda); torch.cuda.init(); print(\'GPU:\', torch.cuda.get_device_name(0))"], app=True)\n    print(t("setup.copying_models"))\n    sync_models(config)\n    install_cloudflared()\n    private_write(CONFIG, json.dumps(config))\n    checkpoint(config, force=True)\n    print(t("setup.done") + root_name + "/models")\n    return config\n\n\ndef refuse_unfinished_update():\n    """A ComfyUI source and packages of two different versions must not be saved or started."""\n    if (PRIVATE / "update-incomplete.json").exists():\n        raise RuntimeError(t("update.incomplete"))\n\n\ndef update_comfyui(config, ref):\n    """Move ComfyUI to another version and install what that version requires.\n\n    Custom nodes, settings, workflows and the database are not files of the ComfyUI\n    repository, so they stay as they are. Drive receives the environment only once the new\n    version is installed and consistent; an update that fails is undone."""\n    if (PRIVATE / "processes.json").exists():\n        raise RuntimeError(t("update.running"))\n    if not re.fullmatch(r"[A-Za-z0-9._/-]+", ref) or ref.startswith("-"):\n        raise ValueError(t("setup.invalid_ref"))\n    drive_root(config)  # The result is saved at the end: find out now if Drive is gone.\n    repo = BASE / "environment/ComfyUI"\n    py = BASE / ".venv/bin/python"\n    lock = "environment/requirements.lock.txt"\n    marker = PRIVATE / "update-incomplete.json"\n\n    def git(*args):\n        return run(["git", *args], app=True, capture=True, cwd=repo, timeout=600).strip()\n\n    def commit(name):\n        # The repository belongs to the application user: take nothing but a commit id from it.\n        try:\n            found = git("rev-parse", "--verify", "--quiet", name + "^{commit}")\n        except RuntimeError:\n            return ""\n        return found if re.fullmatch(r"[0-9a-f]{40,64}", found) else ""\n\n    def switch(target):\n        git("checkout", "--force", "--detach", target)\n        app_action("prepare")  # The checkout puts back the models folder ComfyUI ships.\n\n    def package_names(listing):\n        return {re.split(r"==| @ ", line)[0] for line in listing.splitlines() if LOCK_LINE.fullmatch(line)}\n\n    def undo(state):\n        """Back to the version and the packages recorded before the update began."""\n        packages = checked_lock(app_action("get", lock).decode())\n        index = checked_torch_index(json.loads(app_action("get", "environment/runtime.json")).get("torch_index_url"))\n        switch(state["commit"])\n        install_packages("--index-strategy", "unsafe-best-match", "--extra-index-url", index, "-r", BASE / lock)\n        added = sorted(package_names(run([py, "-m", "pip", "freeze"], app=True, capture=True, timeout=300))\n                       - package_names(packages))\n        if added:\n            run([py, "-m", "uv", "pip", "uninstall", "--python", py, *added], app=True)\n        marker.unlink()\n\n    if marker.exists():\n        # An earlier update failed and could not be undone then.\n        print(t("update.resuming"))\n        undo(json.loads(marker.read_text()))\n    git("fetch", "--tags", "--force", "origin")\n    state = {"commit": commit("HEAD"), "version": git("describe", "--tags", "--always")[:80]}\n    target = next(filter(None, map(commit, ("refs/tags/" + ref, "refs/remotes/origin/" + ref, ref))), "")\n    if not state["commit"] or not target:\n        raise ValueError(t("update.unknown_ref", ref=ref))\n    if target == state["commit"]:\n        print(t("update.already", version=state["version"]))\n        return False\n    print(t("update.starting", previous=state["version"], ref=ref))\n    write_lock()\n    private_write(marker, json.dumps(state))\n    try:\n        switch(target)\n        install_packages(*(argument for name in ("requirements.txt", "manager_requirements.txt")\n                           if (repo / name).exists() for argument in ("-r", repo / name)))\n        run([py, "-m", "pip", "check"], app=True)\n    except Exception:\n        try:\n            undo(state)\n        except Exception:\n            print(t("update.undo_failed"))\n            raise\n        print(t("update.undone", version=state["version"]))\n        raise\n    marker.unlink()\n    version = git("describe", "--tags", "--always")[:80]\n    checkpoint(config, force=True)\n    print(t("update.done", previous=state["version"], version=version))\n    return True\n\n\ndef model_download_url(source, *, hf_repo="", hf_file="", hf_revision="main", civitai_version=""):\n    if source == "Hugging Face":\n        if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", hf_repo):\n            raise ValueError(t("download.hf_repository"))\n        file = safe_relative(hf_file)\n        if not hf_revision or "\\x00" in hf_revision:\n            raise ValueError(t("download.hf_revision"))\n        return f"https://huggingface.co/{hf_repo}/resolve/{quote(hf_revision, safe=\'\')}/{quote(file, safe=\'/\')}"\n    if source == "Civitai":\n        if not re.fullmatch(r"[1-9][0-9]*", str(civitai_version)):\n            raise ValueError(t("download.civitai_version"))\n        return "https://civitai.com/api/download/models/" + str(civitai_version)\n    raise ValueError(t("download.unknown_source"))\n\n\ndef parse_model_url(url):\n    """Download URL and file name for a Hugging Face or Civitai link copied from a browser."""\n    parsed = urlsplit(url.strip())\n    parts = [unquote(part) for part in parsed.path.split("/") if part]\n    host = (parsed.hostname or "").removeprefix("www.")\n    if parsed.scheme == "https" and host == "huggingface.co":\n        if len(parts) < 5 or parts[2] not in {"resolve", "blob"}:\n            raise ValueError(t("download.hf_url"))\n        return model_download_url("Hugging Face", hf_repo="/".join(parts[:2]), hf_file="/".join(parts[4:]),\n                                  hf_revision=parts[3]), parts[-1]\n    if parsed.scheme == "https" and host == "civitai.com":\n        version = parse_qs(parsed.query).get("modelVersionId", [""])[0]\n        if parts[:3] == ["api", "download", "models"] and len(parts) == 4:\n            version = parts[3]\n        elif parts[:1] != ["models"] or not version:\n            raise ValueError(t("download.civitai_url"))\n        return model_download_url("Civitai", civitai_version=version), ""\n    raise ValueError(t("download.unsupported_url"))\n\n\ndef model_token_name(url):\n    """The Colab secret holding the access token for the site a model link points at."""\n    return "CIVITAI_TOKEN" if "civitai.com" in (urlsplit(url.strip()).hostname or "") else "HF_TOKEN"\n\n\ndef served_filename(headers):\n    """The file name a download response declares, or "" unless it is a plain name."""\n    from email.message import EmailMessage\n    message = EmailMessage()\n    try:\n        message["Content-Disposition"] = headers.get("Content-Disposition") or "attachment"\n        name = message.get_filename() or ""\n    except (ValueError, TypeError, LookupError):\n        return ""\n    name = name.replace("\\\\", "/").rsplit("/", 1)[-1]\n    return name if re.fullmatch(r"[^\\x00-\\x1f:]{1,200}", name) and not name.startswith(".") else ""\n\n\ndef fetch_model(config, url, folder, filename="", sha256="", token_secret=""):\n    """Download the model a pasted link points at into a folder of Drive\'s models directory."""\n    download, suggested = parse_model_url(url)\n    folder = safe_relative(folder.strip().strip("/"))\n    filename = filename.strip() or suggested\n    if "/" in filename:\n        raise ValueError(t("download.filename_has_folder"))\n    if filename:\n        return download_model(config, download, folder + "/" + filename, sha256.strip(), token_secret)\n    return download_model(config, download, folder, sha256.strip(), token_secret, name_from_server=True)\n\n\ndef checked_https_url(url):\n    parsed = urlsplit(url)\n    if parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password or parsed.port not in {None, 443}:\n        raise ValueError(t("download.not_https"))\n    import ipaddress\n    import socket\n    if parsed.hostname in {"localhost", "metadata.google.internal"}:\n        raise ValueError(t("download.local_redirect"))\n    try:\n        addresses = {ipaddress.ip_address(parsed.hostname)}\n    except ValueError:\n        # A public-looking name can still point at the VM\'s internal services.\n        try:\n            found = socket.getaddrinfo(parsed.hostname, 443, type=socket.SOCK_STREAM)\n        except OSError:\n            raise ValueError(t("download.unresolved_host")) from None\n        addresses = {ipaddress.ip_address(item[4][0].split("%")[0]) for item in found}\n    if any(not ip.is_global for ip in addresses):\n        raise ValueError(t("download.private_redirect"))\n    return parsed\n\n\ndef stream_model(session, url, target, token_secret=""):\n    """Follow HTTPS redirects, forwarding the token only to the original host."""\n    initial_host = checked_https_url(url).hostname\n    current = url\n    for _ in range(11):\n        parsed = checked_https_url(current)\n        headers = {"User-Agent": "ComfyUI-Colab-Downloader/1"}\n        if token_secret and parsed.hostname == initial_host:\n            headers["Authorization"] = "Bearer " + token_secret\n        with session.get(current, headers=headers, stream=True, timeout=(30, 300), allow_redirects=False) as response:\n            if response.status_code in {301, 302, 303, 307, 308}:\n                location = response.headers.get("Location")\n                if not location:\n                    raise RuntimeError(t("download.redirect_without_target"))\n                current = urljoin(current, location)\n                continue\n            if response.status_code != 200:\n                raise RuntimeError(t("download.http_error", status=response.status_code))\n            content_type = response.headers.get("Content-Type", "").lower()\n            if "text/html" in content_type or "application/json" in content_type:\n                raise RuntimeError(t("download.not_a_model"))\n            digest = hashlib.sha256()\n            with Path(target).open("wb") as file:\n                for chunk in response.iter_content(4 * 1024 * 1024):\n                    file.write(chunk)\n                    digest.update(chunk)\n            if not Path(target).stat().st_size:\n                raise RuntimeError(t("download.empty"))\n            return digest.hexdigest(), served_filename(response.headers)\n    raise RuntimeError(t("download.too_many_redirects"))\n\n\ndef download_model(config, url, relative_path, sha256="", token_secret="", name_from_server=False):\n    """Save a model under relative_path in Drive\'s models folder. With name_from_server,\n    relative_path is the folder and the file takes the name the site serves it under."""\n    import requests\n    rel = safe_relative(relative_path)\n    parsed = checked_https_url(url)\n    if parsed.hostname not in {"huggingface.co", "civitai.com"}:\n        raise ValueError(t("download.host_not_allowed"))\n    if parsed.query or parsed.fragment:\n        raise ValueError(t("download.url_has_query"))\n    if sha256 and not re.fullmatch(r"[a-fA-F0-9]{64}", sha256):\n        raise ValueError(t("download.invalid_sha256"))\n    models = drive_root(config) / "models"\n    models.mkdir(exist_ok=True)\n    target = contained_path(models, rel)\n    if not name_from_server and target.exists():\n        raise FileExistsError(t("download.exists"))\n    folder = target if name_from_server else target.parent\n    folder.mkdir(parents=True, exist_ok=True)\n    partial = folder / (".comfy-" + secrets.token_hex(8) + ".part")\n    try:\n        with requests.Session() as session:\n            session.trust_env = False  # No implicit .netrc credentials or proxy settings.\n            digest, served = stream_model(session, url, partial, token_secret)\n        if sha256 and not hmac.compare_digest(digest, sha256.lower()):\n            raise RuntimeError(t("download.sha256_mismatch"))\n        if name_from_server:\n            if not served:\n                raise RuntimeError(t("download.no_filename"))\n            rel = rel + "/" + served\n            target = contained_path(models, rel)\n        if target.exists():\n            raise FileExistsError(t("download.exists_not_overwritten"))\n        os.replace(partial, target)\n        sync_models(config)\n        print(t("download.saved"), rel, "SHA256:", digest)\n    except requests.RequestException:\n        raise RuntimeError(t("download.connection_failed")) from None\n    finally:\n        partial.unlink(missing_ok=True)\n\n\nSTOP_PATH = "/_colab/stop"\nSTOP_SCRIPT_PATH = "/_colab/stop.js"\n# Added to the ComfyUI page by the proxy: a small button in the tab row, and the dialog it opens.\nSTOP_SCRIPT = r"""(() => {\n  if (window.top !== window || document.getElementById("colab-stop-button")) return;\n  const text = __TEXTS__;\n  const style = document.createElement("style");\n  style.textContent = `\n    #colab-stop-button { position: fixed; top: 7px; right: 56px; z-index: 1000; font: 12px/1 system-ui, sans-serif;\n      padding: 5px 10px; border-radius: 6px; border: 1px solid rgba(128, 128, 128, .5); background: rgba(30, 30, 30, .7);\n      color: #ddd; cursor: pointer; opacity: .6; }\n    #colab-stop-button:hover, #colab-stop-button:focus-visible { opacity: 1; color: #fff; border-color: #f87171; }\n    #colab-stop-dialog { font: 14px/1.6 system-ui, sans-serif; width: min(26rem, calc(100vw - 2rem)); padding: 1.25rem;\n      border: 1px solid #555; border-radius: 10px; background: #202020; color: #eee;\n      box-shadow: 0 12px 40px rgba(0, 0, 0, .6); }\n    #colab-stop-dialog::backdrop { background: rgba(0, 0, 0, .6); }\n    #colab-stop-dialog h2 { margin: 0 0 .5rem; font-size: 1.1rem; }\n    #colab-stop-dialog p { margin: 0 0 .75rem; }\n    #colab-stop-dialog .colab-stop-status { color: #fcd34d; }\n    #colab-stop-dialog .colab-stop-status:empty { display: none; }\n    #colab-stop-dialog .colab-stop-actions { display: flex; flex-direction: column; gap: .5rem; }\n    #colab-stop-dialog button { font: inherit; padding: .55rem .75rem; border-radius: 6px; border: 1px solid #666;\n      background: #333; color: #eee; cursor: pointer; }\n    #colab-stop-dialog button:hover:enabled { background: #444; }\n    #colab-stop-dialog button:disabled { opacity: .45; cursor: default; }\n    #colab-stop-dialog button.colab-stop-danger { background: #b91c1c; border-color: #b91c1c; color: #fff; }\n    #colab-stop-dialog button.colab-stop-danger:hover:enabled { background: #dc2626; }\n    #colab-stop-dialog button.colab-stop-quiet { background: transparent; }\n  `;\n  const add = (parent, tag, content, name) => {\n    const element = document.createElement(tag);\n    if (content) element.textContent = content;\n    if (name) element.className = name;\n    if (tag === "button") element.type = "button";\n    return parent.appendChild(element);\n  };\n  const dialog = document.createElement("dialog");\n  dialog.id = "colab-stop-dialog";\n  add(dialog, "h2", text.title);\n  add(dialog, "p", text.explain);\n  const status = add(dialog, "p", "", "colab-stop-status");\n  const actions = add(dialog, "div", "", "colab-stop-actions");\n  const remove = add(actions, "button", text.delete, "colab-stop-danger");\n  const disconnect = add(actions, "button", text.disconnect);\n  const cancel = add(actions, "button", text.cancel, "colab-stop-quiet");\n  const button = document.createElement("button");\n  button.id = "colab-stop-button";\n  button.type = "button";\n  button.textContent = text.stop;\n  button.title = text.title;\n  let busy = false;\n  const lock = (locked) => {\n    busy = locked;\n    for (const choice of [remove, disconnect, cancel]) choice.disabled = locked;\n  };\n  const ask = async (deleteRuntime) => {\n    lock(true);\n    status.textContent = text.working;\n    try {\n      const answer = await fetch("__STOP_PATH__", {method: "POST", headers: {"Content-Type": "application/json"},\n                                               body: JSON.stringify({delete_runtime: deleteRuntime})});\n      if (!answer.ok) throw new Error(answer.status);\n    } catch (error) {\n      status.textContent = text.failed;\n      lock(false);\n      return;\n    }\n    const started = Date.now();\n    const timer = setInterval(async () => {\n      let state;\n      try {\n        const answer = await fetch("__STOP_PATH__", {cache: "no-store"});\n        if (!answer.ok) throw new Error(answer.status);\n        state = (await answer.json()).state;\n      } catch (error) {\n        // Nothing answers any more: the proxy and the tunnel are down.\n        clearInterval(timer);\n        status.textContent = deleteRuntime ? text.deleted : text.stopped;\n        button.remove();\n        return;\n      }\n      if (state === "failed") {\n        clearInterval(timer);\n        status.textContent = text.failed;\n        lock(false);\n      } else if (Date.now() - started > 180000) {\n        status.textContent = text.slow;\n      }\n    }, 3000);\n  };\n  remove.addEventListener("click", () => ask(true));\n  disconnect.addEventListener("click", () => ask(false));\n  cancel.addEventListener("click", () => dialog.close());\n  dialog.addEventListener("cancel", (event) => { if (busy) event.preventDefault(); });\n  // Keys pressed in the dialog are not ComfyUI shortcuts.\n  dialog.addEventListener("keydown", (event) => event.stopPropagation());\n  button.addEventListener("click", () => {\n    if (!busy) status.textContent = "";\n    dialog.showModal();\n  });\n  const mount = () => {\n    document.head.appendChild(style);\n    document.body.append(button, dialog);\n  };\n  if (document.body) mount(); else document.addEventListener("DOMContentLoaded", mount);\n})();\n"""\n\n\ndef with_stop_button(page):\n    """The ComfyUI page with the script that adds the stop button (unchanged if it has no head)."""\n    position = page.lower().rfind(b"</head>")\n    tag = b\'<script defer src="\' + STOP_SCRIPT_PATH.encode() + b\'"></script>\'\n    return page if position < 0 else page[:position] + tag + page[position:]\n\n\ndef stop_button_script():\n    texts = {"stop": t("menu.stop"), "title": t("menu.title"), "explain": t("menu.explain"), "delete": t("menu.delete"),\n             "disconnect": t("menu.disconnect"), "cancel": t("menu.cancel"), "working": t("menu.working"),\n             "stopped": t("menu.stopped"), "deleted": t("menu.deleted"), "failed": t("menu.failed"), "slow": t("menu.slow")}\n    return STOP_SCRIPT.replace("__TEXTS__", json.dumps(texts)).replace("__STOP_PATH__", STOP_PATH)\n\n\ndef request_stop(delete_runtime):\n    """Leave word, from the proxy, that the ComfyUI page asked to stop. The notebook\'s own\n    process carries it out: only it can save to Drive and give the runtime back to Colab."""\n    (PRIVATE / "stop-status.json").unlink(missing_ok=True)\n    private_write(PRIVATE / "stop-request.json", json.dumps({"delete_runtime": bool(delete_runtime)}))\n\n\ndef stop_request_state():\n    if (PRIVATE / "stop-request.json").exists():\n        return "pending"\n    return "failed" if (PRIVATE / "stop-status.json").exists() else "idle"\n\n\ndef delete_colab_runtime():\n    """Finish writing to Drive, then give the runtime back: it stops using compute units."""\n    from google.colab import drive, runtime as colab\n    drive.flush_and_unmount()\n    colab.unassign()\n\n\ndef carry_out_stop_request(config):\n    """Do what the stop button of the ComfyUI page asked for, if it asked. True once stopped."""\n    request = PRIVATE / "stop-request.json"\n    try:\n        wanted = json.loads(request.read_text())\n    except (OSError, ValueError):\n        return False\n    try:\n        stop(config, save=True)\n    except Exception as failure:\n        # Nothing was stopped, so the page is still there to say that it did not work.\n        private_write(PRIVATE / "stop-status.json", json.dumps({"state": "failed"}))\n        request.unlink(missing_ok=True)\n        print(t("menu.failed"), failure, flush=True)\n        return False\n    if wanted.get("delete_runtime") is True:\n        delete_colab_runtime()\n    return True\n\n\nSTOP_WATCH = []\nSTOP_WATCH_SECONDS = 2\n\n\ndef follow_stop_requests(config):\n    """Watch for the stop button from the notebook\'s process, until stopped one way or another."""\n    cancelled = threading.Event()\n    STOP_WATCH[:] = [cancelled]\n\n    def watch():\n        while not cancelled.wait(STOP_WATCH_SECONDS):\n            if carry_out_stop_request(config):\n                return\n    threading.Thread(target=watch, daemon=True).start()\n\n\nPAIRING_PAGE = """<!doctype html>\n<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n<title>{title}</title>\n<style>\nbody {{ font-family: system-ui, sans-serif; background: #202020; color: #eee; margin: 0;\n       min-height: 100vh; display: flex; align-items: center; justify-content: center; }}\nmain {{ width: 20rem; padding: 1.5rem; }}\nh1 {{ font-size: 1.3rem; }}\ninput, button {{ font: inherit; width: 100%; box-sizing: border-box; padding: .6rem; border-radius: .4rem; }}\ninput {{ font-size: 1.4rem; letter-spacing: .2em; text-align: center; background: #111; color: #eee; border: 1px solid #666; }}\nbutton {{ margin-top: .8rem; background: #3b82f6; color: #fff; border: 0; }}\n.notice {{ color: #fca5a5; }}\n</style></head>\n<body><main><h1>{title}</h1><p>{prompt}</p><p class="notice">{notice}</p>\n<form method="post" action="{path}">\n<input name="code" inputmode="numeric" autocomplete="one-time-code" maxlength="16" autofocus required>\n<button type="submit">{button}</button>\n</form></main></body></html>\n"""\n\n\ndef new_pairing_code():\n    """Issue the code that lets one browser in. It replaces any earlier one."""\n    code = "".join(secrets.choice("0123456789") for _ in range(8))\n    private_write(PRIVATE / "pairing.json", json.dumps(\n        {"code": code, "expires": time.time() + PAIRING_SECONDS, "attempts": PAIRING_ATTEMPTS}))\n    return code\n\n\ndef redeem_pairing_code(supplied):\n    """(accepted, tries left). A code is good for one browser, a few minutes and a few wrong\n    guesses; after any of these there is no code until the notebook issues another."""\n    path = PRIVATE / "pairing.json"\n    try:\n        state = json.loads(path.read_text())\n    except (OSError, ValueError):\n        return False, 0\n    if time.time() > state["expires"]:\n        path.unlink(missing_ok=True)\n        return False, 0\n    if hmac.compare_digest(re.sub(r"[\\s-]", "", supplied).encode(), state["code"].encode()):\n        path.unlink(missing_ok=True)\n        return True, 0\n    state["attempts"] -= 1\n    if state["attempts"] <= 0:\n        path.unlink(missing_ok=True)\n        return False, 0\n    private_write(path, json.dumps(state))\n    return False, state["attempts"]\n\n\ndef create_proxy(config, upstream="http://127.0.0.1:8188", secure_cookie=True):\n    """Streaming proxy in front of ComfyUI.\n\n    Behind the tunnel it lets in only browsers that hold the session cookie, which a pairing\n    code buys once; the cookie also rides Desktop\'s download session. Behind Colab\'s port\n    forwarding Google has already let in the runtime\'s owner only, so it just keeps other\n    sites from sending requests through that browser. Behind Cloudflare Access it lets in\n    only requests that carry Access\'s signature for this application."""\n    from aiohttp import web, ClientError, ClientSession, ClientTimeout, WSMsgType, DummyCookieJar\n    import html\n    forwarded = config.get("mode") == "colab"\n    access = config.get("access") if config.get("mode") == "access" else None\n    if access:\n        import jwt\n    signing_keys = {}\n    keys_fetched = [float("-inf")]\n    verified = {}  # token -> expiry: one page load sends the same token a few hundred times.\n    cookie_secret = config.get("cookie_secret", "")\n    cookie_name = COOKIE if secure_cookie else "comfy_colab"\n    origins = set(config.get("origins", ()))\n    hop = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization",\n           "te", "trailer", "transfer-encoding", "upgrade", "content-length"}\n    client_key = web.AppKey("client", ClientSession)\n    # Vite names frontend files after their content, so they can be cached for good.\n    hashed_asset = re.compile(r"/assets/[\\w.-]+-[A-Za-z0-9_-]{8}\\.(?:js|css|json|svg|ttf|woff2?)")\n    compressible = ("text/", "application/javascript", "application/json", "image/svg+xml")\n\n    def matches(supplied, secret):\n        # Bytes: comparing str raises on the non-ASCII input anyone can send.\n        return bool(secret) and hmac.compare_digest(supplied.encode(), secret.encode())\n\n    async def lifecycle(app):\n        app[client_key] = ClientSession(timeout=ClientTimeout(total=None, sock_connect=30),\n                                      auto_decompress=False, cookie_jar=DummyCookieJar())\n        yield\n        await app[client_key].close()\n\n    async def signing_key(kid):\n        if kid not in signing_keys and time.monotonic() - keys_fetched[0] >= ACCESS_KEYS_REFRESH:\n            # Cloudflare rotates its keys, so an unknown one is looked up, but not for every bad request.\n            keys_fetched[0] = time.monotonic()\n            async with ClientSession(timeout=ClientTimeout(total=15)) as session:\n                async with session.get(access_certs_url(access["team"])) as res:\n                    published = (await res.json(content_type=None))["keys"]\n            signing_keys.update({key["kid"]: jwt.PyJWK.from_dict(key).key for key in published})\n        return signing_keys[kid]\n\n    async def signed_by_access(request):\n        """Cloudflare adds this token to what it lets through. Checking it here means a request\n        that reaches the tunnel some other way, or for another application, is still refused."""\n        token = request.headers.get("Cf-Access-Jwt-Assertion", "")\n        if not token or len(token) > 8192:\n            return False\n        if verified.get(token, 0) > time.time():\n            return True\n        try:\n            key = await signing_key(jwt.get_unverified_header(token)["kid"])\n            claims = jwt.decode(token, key, algorithms=["RS256"], audience=access["aud"],\n                                issuer="https://" + access["team"], options={"require": ["exp", "iat", "aud", "iss"]})\n        except (jwt.PyJWTError, ClientError, KeyError, TypeError, ValueError, OSError, asyncio.TimeoutError):\n            return False\n        if len(verified) >= 256:\n            verified.clear()\n        verified[token] = claims["exp"]\n        return True\n\n    def refuse_other_sites(request):\n        origin = request.headers.get("Origin")\n        if origin and origin not in {"https://" + request.host, "http://" + request.host} | origins:\n            print(t("proxy.origin_rejected", origin=origin[:200], host=request.host[:200]), flush=True)\n            raise web.HTTPForbidden(text=t("proxy.cross_origin"))\n        return origin\n\n    def pairing_page(notice="", status=200):\n        page = PAIRING_PAGE.format(path=PAIRING_PATH, notice=html.escape(notice), title=html.escape(t("pairing.title")),\n                                   prompt=html.escape(t("pairing.prompt")), button=html.escape(t("pairing.button")))\n        # Not no-referrer: a browser then submits the form with "Origin: null", which is refused below.\n        return web.Response(text=page, status=status, content_type="text/html", headers={\n            "Cache-Control": "no-store", "Referrer-Policy": "same-origin", "X-Frame-Options": "DENY",\n            "Content-Security-Policy": "default-src \'none\'; style-src \'unsafe-inline\'; form-action \'self\'; "\n                                       "frame-ancestors \'none\'; base-uri \'none\'"})\n\n    async def pair(request):\n        """The one page anyone can reach: it trades the code shown in the notebook for the cookie."""\n        if request.method != "POST":\n            return pairing_page()\n        refuse_other_sites(request)\n        length = request.content_length\n        if length is None or length > 256:  # Nothing unauthenticated gets to send more.\n            raise web.HTTPBadRequest()\n        try:\n            form = parse_qs((await request.content.readexactly(length)).decode(errors="replace"))\n        except asyncio.IncompleteReadError:\n            raise web.HTTPBadRequest() from None\n        accepted, remaining = redeem_pairing_code(form.get("code", [""])[0])\n        if not accepted:\n            return pairing_page(t("pairing.wrong", remaining=remaining) if remaining else t("pairing.closed"), 401)\n        response = web.HTTPSeeOther(location="/")\n        response.set_cookie(cookie_name, cookie_secret, max_age=COOKIE_SECONDS,\n                            secure=secure_cookie, httponly=True, samesite="Lax", path="/")\n        response.headers["Cache-Control"] = "no-store"\n        response.headers["Referrer-Policy"] = "no-referrer"\n        raise response\n\n    async def stop_endpoint(request):\n        """The stop button\'s request, and afterwards its questions about how that is going."""\n        if request.method == "POST":\n            length = request.content_length\n            if length is None or length > 256:\n                raise web.HTTPBadRequest()\n            try:\n                delete_runtime = json.loads(await request.content.readexactly(length))["delete_runtime"]\n            except (asyncio.IncompleteReadError, ValueError, KeyError, TypeError):\n                raise web.HTTPBadRequest() from None\n            if not isinstance(delete_runtime, bool):\n                raise web.HTTPBadRequest()\n            request_stop(delete_runtime)\n        return web.json_response({"state": stop_request_state()}, status=202 if request.method == "POST" else 200,\n                                 headers={"Cache-Control": "no-store"})\n\n    async def handler(request):\n        if access:\n            if not await signed_by_access(request):\n                raise web.HTTPUnauthorized(text=t("proxy.access_required"))\n        elif not forwarded and not matches(request.cookies.get(cookie_name, ""), cookie_secret):\n            if request.path == PAIRING_PATH:\n                return await pair(request)\n            if request.method == "GET" and "text/html" in request.headers.get("Accept", ""):\n                # A browser opening the address: take it to where the code goes.\n                raise web.HTTPSeeOther(location=PAIRING_PATH, headers={"Cache-Control": "no-store"})\n            raise web.HTTPUnauthorized(text=t("proxy.unauthorized"))\n        if request.path == PAIRING_PATH:\n            raise web.HTTPSeeOther(location="/")\n        origin = refuse_other_sites(request)\n        if request.path == STOP_SCRIPT_PATH:\n            return web.Response(text=stop_button_script(), content_type="application/javascript", charset="utf-8",\n                                headers={"Cache-Control": "no-store"})\n        if request.path == STOP_PATH:\n            return await stop_endpoint(request)\n        # What proves the visitor\'s identity stops here: custom nodes get neither the cookie nor Access\'s token.\n        headers = {k: v for k, v in request.headers.items()\n                   if k.lower() not in hop | {"cookie", "authorization"} and not k.lower().startswith("cf-access-")}\n        if origin:\n            # ComfyUI refuses a request whose Host and Origin differ, and a forwarder may rewrite Host.\n            headers = {k: v for k, v in headers.items() if k.lower() != "host"}\n            headers["Host"] = urlsplit(origin).netloc\n        # ComfyUI answers 403 to cross-site requests, which includes following the\n        # connection URL from a link (e.g. Colab output). Loading the root page is safe.\n        if request.method == "GET" and request.path == "/":\n            headers = {k: v for k, v in headers.items() if k.lower() != "sec-fetch-site"}\n        page = request.method == "GET" and request.path in ("/", "/index.html")\n        if page:\n            # Asked for uncompressed: the stop button goes into this page before it is passed on.\n            headers = {k: v for k, v in headers.items() if k.lower() != "accept-encoding"}\n            headers["Accept-Encoding"] = "identity"\n        # Preserve other backend cookies, but never disclose the proxy credential to custom nodes.\n        # Behind Colab\'s forwarding its own sign-in cookies cannot be told apart, so none are passed on.\n        backend_cookies = {} if forwarded else {\n            k: v for k, v in request.cookies.items() if k not in {COOKIE, "comfy_colab"} and not k.startswith("CF_")}\n        if backend_cookies:\n            from http.cookies import SimpleCookie\n            cookies = SimpleCookie()\n            for k, v in backend_cookies.items():\n                cookies[k] = v\n            headers["Cookie"] = cookies.output(header="", sep=";").strip()\n        url = upstream + request.rel_url.raw_path_qs\n        if request.headers.get("Upgrade", "").lower() == "websocket":\n            # Each hop negotiates its own handshake. Forwarding the browser\'s\n            # permessage-deflate offer makes ComfyUI compress frames this client cannot read.\n            ws_headers = {k: v for k, v in headers.items() if not k.lower().startswith("sec-websocket-")}\n            protocols = [p.strip() for p in request.headers.get("Sec-WebSocket-Protocol", "").split(",") if p.strip()]\n            async with request.app[client_key].ws_connect(url, headers=ws_headers, protocols=protocols,\n                                                        max_msg_size=0) as remote:\n                local = web.WebSocketResponse(max_msg_size=0, protocols=[remote.protocol] if remote.protocol else ())\n                await local.prepare(request)\n\n                async def relay(source, dest):\n                    async for msg in source:\n                        if msg.type == WSMsgType.TEXT:\n                            await dest.send_str(msg.data)\n                        elif msg.type == WSMsgType.BINARY:\n                            await dest.send_bytes(msg.data)\n                        elif msg.type in {WSMsgType.CLOSE, WSMsgType.CLOSED, WSMsgType.ERROR}:\n                            if msg.type == WSMsgType.ERROR:\n                                print(t("proxy.websocket_error"), repr(source.exception()), flush=True)\n                            break\n                tasks = [asyncio.create_task(relay(local, remote)), asyncio.create_task(relay(remote, local))]\n                try:\n                    await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)\n                finally:\n                    for task in tasks:\n                        task.cancel()\n                    await asyncio.gather(*tasks, return_exceptions=True)\n                    await local.close()\n                return local\n        try:\n            async with request.app[client_key].request(request.method, url, headers=headers,\n                    data=request.content.iter_chunked(1024 * 1024), allow_redirects=False) as res:\n                page = (page and res.status == 200 and "Content-Encoding" not in res.headers\n                        and res.headers.get("Content-Type", "").lower().startswith("text/html"))\n                if page:\n                    response = web.Response(body=with_stop_button(await res.read()), headers=[\n                        (k, v) for k, v in res.headers.items() if k.lower() not in hop])\n                else:\n                    response = web.StreamResponse(status=res.status, headers=[\n                        (k, v) for k, v in res.headers.items() if k.lower() not in hop - {"content-length"}])\n                response.headers["Referrer-Policy"] = "no-referrer"\n                complete = request.method == "GET" and res.status == 200\n                response.headers["Cache-Control"] = ("private, max-age=31536000, immutable"\n                    if complete and hashed_asset.fullmatch(request.path) else "no-store")\n                # ComfyUI sends its 26 MB frontend uncompressed and the tunnel is the slow hop.\n                if (complete and "Content-Encoding" not in res.headers\n                        and res.headers.get("Content-Type", "").lower().startswith(compressible)\n                        and "gzip" in request.headers.get("Accept-Encoding", "").lower()):\n                    response.headers["Vary"] = "Accept-Encoding"\n                    response.enable_compression(web.ContentCoding.gzip)\n                if page:\n                    return response\n                await response.prepare(request)\n                async for chunk in res.content.iter_chunked(1024 * 1024):\n                    await response.write(chunk)\n                await response.write_eof()\n                return response\n        except (ConnectionError, OSError):\n            raise web.HTTPBadGateway(text=t("proxy.upstream_down")) from None\n\n    app = web.Application(client_max_size=1024 ** 3)\n    app.cleanup_ctx.append(lifecycle)\n    app.router.add_route("*", "/{path:.*}", handler)\n    return app\n\n\n# What the frontend awaits before it removes its splash logo.\nSTARTUP_ROUTES = ("/", "/api/features", "/api/users", "/api/settings", "/api/i18n", "/api/extensions",\n                  "/api/object_info", "/api/userdata?dir=workflows&recurse=true&split=false&full_info=true")\n\n\nasync def probe_frontend(base, headers=None, timeout=30):\n    """Request the frontend\'s startup dependencies at base; print each result and return the failure count."""\n    from aiohttp import ClientSession, ClientTimeout, DummyCookieJar, WSMsgType\n    failures = 0\n    compressed = []\n\n    async def fetch(session, route):\n        started = time.monotonic()\n        try:\n            async with session.get(base + route, headers=headers, allow_redirects=False) as res:\n                body = await res.read()\n                compressed.append("Content-Encoding" in res.headers)\n                return res.status, body, time.monotonic() - started\n        except Exception as exc:\n            return type(exc).__name__, b"", time.monotonic() - started\n\n    def healthy(route, status, seconds):\n        missing_is_normal = status == 404 and "userdata" in route\n        return isinstance(status, int) and (status < 400 or missing_is_normal) and seconds < 10\n\n    async with ClientSession(timeout=ClientTimeout(total=timeout), cookie_jar=DummyCookieJar()) as session:\n        files = set()\n        for route in STARTUP_ROUTES:\n            status, body, seconds = await fetch(session, route)\n            ok = healthy(route, status, seconds)\n            failures += not ok\n            print(t("diagnosis.route", mark="OK" if ok else "NG", status=status, seconds=seconds, size=len(body),\n                    route=route.split("?")[0]))\n            if route == "/":\n                files.update("/" + name.decode() for name in re.findall(rb\'(?:src|href)="\\./(assets/[^"]+)"\', body))\n            elif route == "/api/extensions" and status == 200:\n                # The list comes from ComfyUI; only a path can be appended to the address safely.\n                files.update(quote(name) for name in json.loads(body) if isinstance(name, str) and name.startswith("/"))\n        gate = asyncio.Semaphore(16)\n\n        size = 0\n\n        async def fetch_file(route):\n            nonlocal size\n            async with gate:\n                status, body, seconds = await fetch(session, route)\n                size += len(body)\n                # Follow the chunks each script imports, as the browser does.\n                more = {"/assets/" + name.decode() for name in\n                        re.findall(rb"(?:\\./|assets/)([\\w.-]+-[\\w-]{8}\\.(?:js|css))", body)} if route.startswith("/assets/") else set()\n                return route, status, seconds, more\n        del compressed[:]\n        started = time.monotonic()\n        results, pending, listed = [], sorted(files), set(files)\n        while pending:\n            batch = await asyncio.gather(*map(fetch_file, pending))\n            results += [item[:3] for item in batch]\n            found = set().union(*(item[3] for item in batch)) - files\n            files |= found\n            pending = sorted(found)\n        elapsed = time.monotonic() - started\n        # A name found inside a script may be just text; only a listed file must exist.\n        bad = [item for item in results if not healthy(*item) and (item[0] in listed or item[1] != 404)]\n        failures += len(bad)\n        # The browser downloads these before the UI appears, so this time predicts the wait.\n        print(t("diagnosis.assets", mark="NG" if bad else "OK", total=len(results), failed=len(bad),\n                megabytes=size / 1048576, elapsed=elapsed, compressed=sum(compressed)))\n        for route, status, seconds in bad[:10]:\n            print(t("diagnosis.failed_asset", status=status, seconds=seconds, route=route))\n        try:\n            # Browsers always offer compression; the first frame carries the client id.\n            async with session.ws_connect(base.replace("http", "ws", 1) + "/ws", headers=headers, compress=15) as ws:\n                message = await ws.receive(timeout=10)\n                kind = json.loads(message.data).get("type") if message.type == WSMsgType.TEXT else message.type.name\n        except Exception as exc:\n            kind = type(exc).__name__\n        failures += kind != "status"\n        print(t("diagnosis.websocket", mark="OK" if kind == "status" else "NG", kind=kind))\n    return failures\n\n\nasync def diagnose(config):\n    """Find the hop where the UI stops loading: ComfyUI, the auth proxy, or the tunnel."""\n    if "mode" not in config:\n        raise SystemExit(t("diagnosis.not_running"))\n    auth = {"Cookie": COOKIE + "=" + config["cookie_secret"]} if config.get("cookie_secret") else None\n    targets = [(t("diagnosis.target_comfyui"), "http://127.0.0.1:8188", None)]\n    if config["mode"] != "access":  # There the proxy answers only what Cloudflare Access has signed.\n        targets.append((t("diagnosis.target_proxy"), "http://127.0.0.1:8189", auth))\n    if config.get("remote_url"):\n        targets.append((t("diagnosis.target_public"), config["remote_url"], auth))\n    for label, base, headers in targets:\n        print("== " + label, flush=True)\n        if await probe_frontend(base, headers):\n            print(t("diagnosis.failed_at", label=label))\n            return label\n    # Colab\'s forwarding and Cloudflare Access only answer a signed-in browser, so they cannot be tried from here.\n    if config["mode"] == "colab":\n        print(t("diagnosis.colab_ok"))\n    elif config["mode"] == "access":\n        print(t("diagnosis.access_ok"))\n    else:\n        print(t("diagnosis.all_ok"))\n    return None\n\n\ndef isolation_report():\n    """List the local services the application user can reach. Colab runs its own services as\n    root; one that answers without authentication would let a custom node leave its account."""\n    import socket\n    services = set()\n    for table in ("/proc/net/tcp", "/proc/net/tcp6"):\n        for line in Path(table).read_text().splitlines()[1:]:\n            fields = line.split()\n            address, port = fields[1].rsplit(":", 1)\n            if fields[3] != "0A":  # Not listening.\n                continue\n            host = socket.inet_ntoa(bytes.fromhex(address)[::-1]) if len(address) == 8 else "127.0.0.1"\n            services.add(("127.0.0.1" if host == "0.0.0.0" else host, int(port, 16)))\n    for name, value in sorted(os.environ.items()):\n        match = re.fullmatch(r"(\\d+\\.\\d+\\.\\d+\\.\\d+):(\\d+)", value)\n        if match and re.search(r"CRED|TBE|METADATA", name):\n            services.add((match.group(1), int(match.group(2))))\n    services.add(("169.254.169.254", 80))\n    # Unix sockets that accept connections: by path, or "@name" in the abstract namespace.\n    sockets = sorted({fields[7] for fields in map(str.split, Path("/proc/net/unix").read_text().splitlines()[1:])\n                      if len(fields) > 7 and fields[3] == "00010000"})\n    probe = ("import json, socket, sys\\n"\n             "def ask(host, port, path):\\n"\n             "    with socket.create_connection((host, port), 3) as s:\\n"\n             "        try:\\n"\n             "            s.sendall((\'GET \' + path + \' HTTP/1.0\\\\r\\\\nHost: localhost\\\\r\\\\n\\\\r\\\\n\').encode())\\n"\n             "            return s.recv(64).split(b\'\\\\r\\\\n\')[0].decode(\'latin-1\')\\n"\n             "        except OSError:\\n"\n             "            return \'\'\\n"\n             "tcp, unix = json.load(sys.stdin)\\n"\n             "for host, port in tcp:\\n"\n             "    try:\\n"\n             "        answer = ask(host, port, \'/\') or \'connected, no HTTP reply\'\\n"\n             "    except OSError as exc:\\n"\n             "        answer = \'unreachable (\' + type(exc).__name__ + \')\'\\n"\n             "    if answer.startswith(\'HTTP/\'):\\n"\n             "        try:\\n"\n             "            answer += \' | /api/kernels: \' + ask(host, port, \'/api/kernels\').partition(\' \')[2]\\n"\n             "        except OSError:\\n"\n             "            pass\\n"\n             "    print(f\'tcp\\\\t{host}:{port}\\\\t{answer}\', flush=True)\\n"\n             "for name in unix:\\n"\n             "    s = socket.socket(socket.AF_UNIX)\\n"\n             "    s.settimeout(3)\\n"\n             "    try:\\n"\n             "        s.connect(\'\\\\0\' + name[1:] if name.startswith(\'@\') else name)\\n"\n             "        print(f\'unix\\\\t{name}\\\\tconnected\', flush=True)\\n"\n             "    except OSError:\\n"\n             "        pass\\n"\n             "    s.close()\\n")\n    result = subprocess.run(app_command([sys.executable, "-c", probe]), env=application_environment(),\n                            input=json.dumps([sorted(services), sockets]), capture_output=True, text=True, timeout=600)\n    owners = subprocess.run(["ss", "-H", "-ltnp"], capture_output=True, text=True).stdout if shutil.which("ss") else ""\n    print(t("isolation.heading", user=APP_USER))\n    reachable = []\n    for line in result.stdout.splitlines():\n        kind, _, rest = line.partition("\\t")\n        address, _, answer = rest.partition("\\t")\n        if kind == "unix":\n            reachable.append(address)\n            continue\n        port = address.rsplit(":", 1)[1]\n        owner = sorted(set(re.findall(r\'\\(\\("([^"]+)"\', " ".join(\n            row for row in owners.splitlines() if row.split()[3].rsplit(":", 1)[-1] == port))))\n        note = {"8188": t("isolation.comfyui"), "8189": t("isolation.proxy")}.get(port, ", ".join(owner) or t("common.unknown"))\n        print(f"  {address:<24} {answer:<56} {note}")\n    print(t("isolation.unix_sockets", listening=len(sockets), reachable=len(reachable)), ", ".join(reachable) or t("common.none"))\n    import ctypes\n    abi = ctypes.CDLL(None, use_errno=True).syscall(444, None, 0, 1)  # landlock_create_ruleset(version)\n    lsm = Path("/sys/kernel/security/lsm")\n    print(t("isolation.kernel"), os.uname().release, "| Landlock ABI:", abi if abi > 0 else t("common.none"),\n          "| LSM:", lsm.read_text().strip() if lsm.exists() else t("common.unknown"))\n    effective = int(re.search(r"CapEff:\\s*([0-9a-f]+)", Path("/proc/self/status").read_text()).group(1), 16)\n    print(t("isolation.root_capabilities"), ", ".join(f"{name}={t(\'common.yes\') if effective >> bit & 1 else t(\'common.no\')}" for name, bit in\n                               (("NET_ADMIN", 12), ("SYS_ADMIN", 21), ("SETUID", 7), ("SYS_PTRACE", 19))))\n\n    def usable(command):\n        if not shutil.which(command[0]):\n            return t("isolation.no_command")\n        return t("isolation.available") if not subprocess.run(command, capture_output=True).returncode else t("isolation.unavailable")\n    print("iptables:", usable(["iptables", "-S", "OUTPUT"]), "| network namespace:", usable(["unshare", "--net", "true"]),\n          "| mount namespace:", usable(["unshare", "--mount", "true"]), "| ip:", t("common.present") if shutil.which("ip") else t("common.none"),\n          "| /dev/net/tun:", t("common.present") if Path("/dev/net/tun").exists() else t("common.none"))\n\n\ndef backend_command():\n    repo = BASE / "environment/ComfyUI"\n    return [str(BASE / ".venv/bin/python"), str(repo / "main.py"),\n            "--listen", "127.0.0.1", "--port", "8188", "--enable-manager",\n            "--base-directory", str(repo), "--user-directory", str(repo / "user"),\n            "--input-directory", str(BASE / "input"), "--output-directory", str(BASE / "output"),\n            "--temp-directory", str(BASE / "temp"), "--extra-model-paths-config", str(BASE / "models.yaml"),\n            "--database-url", "sqlite:///" + str(BASE / "database/comfyui.db")]\n\n\ndef supervise(config):\n    # Background process persists independently of the notebook cell.\n    def save_loop():\n        while True:\n            time.sleep(config["checkpoint_seconds"])\n            try:\n                checkpoint(config)\n            except Exception as exc:\n                private_write(PRIVATE / "checkpoint-status.json", json.dumps({"ok": False, "error": str(exc)}))\n                print(t("supervisor.autosave_failed"), flush=True)\n    threading.Thread(target=save_loop, daemon=True).start()\n    env = application_environment()\n    for _ in range(5):\n        with (PRIVATE / "comfy.log").open("a") as log:\n            child = subprocess.Popen(app_command(backend_command()), cwd=BASE / "environment/ComfyUI", env=env,\n                                     stdout=log, stderr=subprocess.STDOUT)\n            child.wait()\n        try:\n            checkpoint(config, force=True)\n        except Exception:\n            print(t("supervisor.save_before_restart_failed"), flush=True)\n        print(t("supervisor.restarting"), flush=True)\n        time.sleep(10)\n    print(t("supervisor.restart_limit"), flush=True)\n\n\ndef redact(text):\n    """Hide credentials and the tunnel address in text that is about to be displayed."""\n    text = re.sub(r"(?i)(access_key=|Bearer\\s+)[A-Za-z0-9_.-]+", r"\\1[REDACTED]", text)\n    text = re.sub(r"hf_[A-Za-z0-9]+", "[REDACTED]", text)\n    text = re.sub(r"eyJ[A-Za-z0-9_+/=-]{20,}(\\.[A-Za-z0-9_+/=-]+)*", "[REDACTED]", text)  # Tunnel and Access tokens.\n    return re.sub(r"[a-z0-9-]+\\.trycloudflare\\.com", "[REDACTED].trycloudflare.com", text)\n\n\ndef colab_forwarded_address(port):\n    """Where Colab serves a port of this runtime, to the browser signed in as its owner only."""\n    from google.colab import output\n    parts = urlsplit(str(output.eval_js(f"google.colab.kernel.proxyPort({port})")))\n    if parts.scheme != "https" or not parts.netloc:\n        raise RuntimeError(t("start.colab_unavailable"))\n    return f"https://{parts.netloc}/"\n\n\ndef show_pairing(config):\n    code = new_pairing_code()\n    print(t("start.tunnel_heading"))\n    print(config["remote_url"] + "/")\n    print(t("start.pairing_code", code=code[:4] + "-" + code[4:], minutes=PAIRING_SECONDS // 60))\n    print(t("start.pairing_note"))\n\n\ndef issue_pairing_code(config):\n    """Let one more browser in while ComfyUI runs behind the tunnel."""\n    if config.get("mode") != "tunnel" or not config.get("remote_url"):\n        raise RuntimeError(t("pairing.not_running"))\n    show_pairing(config)\n\n\ndef access_certs_url(team_domain):\n    return f"https://{team_domain}/cdn-cgi/access/certs"\n\n\ndef access_settings(hostname, audience):\n    """What the notebook was told about the Cloudflare side, checked before anything starts."""\n    hostname = re.sub(r"^https://|/+$", "", hostname.strip().lower())\n    if not re.fullmatch(r"(?=.{4,253}$)([a-z0-9]([a-z0-9-]{0,61}[a-z0-9])?\\.)+[a-z0-9-]{2,63}", hostname):\n        raise ValueError(t("access.invalid_hostname"))\n    audience = (audience or "").strip().lower()\n    if not re.fullmatch(r"[0-9a-f]{64}", audience):\n        raise ValueError(t("access.invalid_audience"))\n    return {"hostname": hostname, "aud": audience}\n\n\ndef access_team_domain(hostname):\n    """The team whose sign-in Cloudflare Access puts in front of the hostname.\n\n    Asked the way a stranger would, over verified HTTPS: only whoever controls the hostname can\n    make it send visitors elsewhere, and then no setting here would help. Without that redirect\n    there is nothing to check signatures against, so nothing is started."""\n    import requests\n    status = "-"\n    for _ in range(4):\n        try:\n            answer = requests.get("https://" + hostname + "/", timeout=10, allow_redirects=False)\n        except requests.RequestException:\n            time.sleep(3)\n            continue\n        sign_in = urlsplit(answer.headers.get("Location", ""))\n        if (answer.status_code in (301, 302, 303, 307, 308) and sign_in.scheme == "https"\n                and re.fullmatch(r"[a-z0-9][a-z0-9-]*\\.cloudflareaccess\\.com", sign_in.netloc)\n                and sign_in.path.startswith("/cdn-cgi/access/login")):\n            return sign_in.netloc\n        status = answer.status_code\n        time.sleep(3)\n    raise RuntimeError(t("access.unprotected", hostname=hostname, status=status))\n\n\ndef tunnel_services(log):\n    """Where the named tunnel sends its hostnames: the services in the latest settings cloudflared\n    reports having received from Cloudflare. None while it has reported none that can be read."""\n    services = None\n    for line in log.splitlines():\n        if "Updated to new configuration" not in line:\n            continue\n        found = re.search(r\'config"?[=:]\\s*"((?:[^"\\\\]|\\\\.)*)"\', line)\n        try:\n            rules = json.loads(json.loads(\'"\' + found.group(1) + \'"\'))["ingress"]\n            services = [str(rule["service"]) for rule in rules]\n        except (AttributeError, ValueError, KeyError, TypeError):\n            services = None\n    return services\n\n\ndef tunnel_leads_to_proxy(services):\n    """Every hostname of the tunnel has to end at the proxy. Any other port of this runtime it\n    named would be served to whoever gets past Cloudflare, without the checks made here."""\n    proxy = [service for service in services if service.rstrip("/") in ("http://127.0.0.1:8189", "http://localhost:8189")]\n    others = [service for service in services if service not in proxy and not re.fullmatch(r"http_status:\\d{3}", service)]\n    return bool(proxy) and not others\n\n\ndef start_with_access(config, hostname, audience, tunnel_token):\n    """Start behind a named Cloudflare tunnel whose hostname Cloudflare Access protects."""\n    access = access_settings(hostname, audience)\n    token = (tunnel_token or "").strip()\n    if not re.fullmatch(r"[A-Za-z0-9+/=_-]{40,4096}", token):\n        raise ValueError(t("access.invalid_token"))\n    if (PRIVATE / "processes.json").exists():\n        raise RuntimeError(t("start.already_running"))\n    access["team"] = access_team_domain(access["hostname"])\n    print(t("access.team", team=access["team"]))\n    start(config, "access", access=access, tunnel_token=token)\n\n\ndef start(config, mode, access=None, tunnel_token=""):\n    """Start ComfyUI and make it reachable: mode "colab" through Colab\'s port forwarding,\n    "tunnel" through a Quick Tunnel that asks each browser for a pairing code, "access"\n    through a named tunnel on a hostname that Cloudflare Access protects."""\n    import requests\n    if mode not in ("colab", "tunnel", "access") or (mode == "access") != bool(access and tunnel_token):\n        raise ValueError(mode)\n    if (PRIVATE / "processes.json").exists():\n        raise RuntimeError(t("start.already_running"))\n    refuse_unfinished_update()\n    for name in SESSION_KEYS:\n        config.pop(name, None)\n    for name in ("stop-request.json", "stop-status.json"):\n        (PRIVATE / name).unlink(missing_ok=True)\n    config["mode"] = mode\n    if mode == "colab":\n        address = colab_forwarded_address(8189)\n        config["origins"] = [address.rstrip("/")]\n    elif mode == "tunnel":\n        # A cookie from an earlier start (browser, Desktop) stops working.\n        config["cookie_secret"] = secrets.token_urlsafe(32)\n    else:\n        config["access"] = access  # Not the tunnel\'s token: only cloudflared is given that.\n    private_write(CONFIG, json.dumps(config))\n    module = Path(__file__).resolve()\n    processes = {}\n\n    def launch(name):\n        with (PRIVATE / (name + ".log")).open("a") as log:\n            process = subprocess.Popen([sys.executable, str(module), name.replace("supervisor", "supervise")],\n                                       stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n        processes[name] = process.pid\n        private_write(PRIVATE / "processes.json", json.dumps(processes))\n        return process\n    try:\n        # A process left behind by a custom node could take the proxy\'s port and receive the tunnel.\n        stop_app_processes()\n        proxy = launch("proxy")\n        for _ in range(30):\n            if proxy.poll() is not None:\n                raise RuntimeError(t("start.proxy_exited_port"))\n            try:\n                # Its answer before ComfyUI runs: no cookie behind the tunnel, no backend behind Colab.\n                if requests.get("http://127.0.0.1:8189/", timeout=3).status_code == (502 if mode == "colab" else 401):\n                    break\n            except requests.RequestException:\n                pass\n            time.sleep(1)\n        else:\n            raise RuntimeError(t("start.proxy_not_started"))\n        launch("supervisor")\n        deadline = time.monotonic() + 300\n        while time.monotonic() < deadline:\n            try:\n                if requests.get("http://127.0.0.1:8188/system_stats", timeout=3).ok:\n                    break\n            except requests.RequestException:\n                pass\n            time.sleep(2)\n        else:\n            raise RuntimeError(t("start.comfyui_timeout"))\n        if proxy.poll() is not None:\n            raise RuntimeError(t("start.proxy_exited"))\n        if mode == "colab":\n            print(t("start.colab_heading"))\n            print(address)\n            print(t("start.colab_note"))\n            follow_stop_requests(config)\n            return\n        logfile = PRIVATE / "tunnel.log"\n        logfile.write_text("")\n        with logfile.open("a") as log:\n            if mode == "access":\n                # The token goes in the environment: a command line can be read by every user here.\n                tunnel = subprocess.Popen([str(PRIVATE / "cloudflared"), "tunnel", "--no-autoupdate", "run"],\n                                          env={**os.environ, "TUNNEL_TOKEN": tunnel_token},\n                                          stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n            else:\n                tunnel = subprocess.Popen([str(PRIVATE / "cloudflared"), "tunnel", "--no-autoupdate",\n                        "--url", "http://127.0.0.1:8189"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n        processes["tunnel"] = tunnel.pid\n        private_write(PRIVATE / "processes.json", json.dumps(processes))\n        if mode == "access":\n            services = connected = None\n            for second in range(60):\n                log = logfile.read_text()\n                services = tunnel_services(log)\n                if connected is None and "Registered tunnel connection" in log:\n                    connected = second\n                # Cloudflare sends the tunnel its settings right after it connects.\n                if services is not None or (connected is not None and second - connected >= 15):\n                    break\n                if tunnel.poll() is not None:\n                    raise RuntimeError(t("access.tunnel_exited"))\n                time.sleep(1)\n            if services is None:\n                print(t("access.target_unknown"))\n            elif not tunnel_leads_to_proxy(services):\n                raise RuntimeError(t("access.wrong_target", services=", ".join(services)[:300]))\n            print(t("access.heading"))\n            print("https://" + access["hostname"] + "/")\n            print(t("access.note"))\n            print(t("start.url_auto_download"))\n            follow_stop_requests(config)\n            return\n        for _ in range(90):\n            match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", logfile.read_text())\n            if match:\n                url = match.group(0)\n                config["remote_url"] = url\n                private_write(CONFIG, json.dumps(config))\n                show_pairing(config)\n                print(t("start.url_auto_download"))\n                follow_stop_requests(config)\n                return\n            if tunnel.poll() is not None:\n                raise RuntimeError(t("start.tunnel_exited"))\n            time.sleep(1)\n        raise RuntimeError(t("start.no_tunnel_url"))\n    except Exception:\n        stop(config, save=False)\n        raise\n\n\ndef stop(config, save=True):\n    if save:\n        checkpoint(config, force=True)  # A failed save must not look like successful shutdown.\n        print(t("stop.saved"))\n    file = PRIVATE / "processes.json"\n    if file.exists():\n        for pid in reversed(list(json.loads(file.read_text()).values())):\n            try:\n                os.killpg(pid, signal.SIGTERM)\n            except ProcessLookupError:\n                pass\n        for pid in json.loads(file.read_text()).values():\n            deadline = time.monotonic() + 5\n            while time.monotonic() < deadline:\n                stat = Path(f"/proc/{pid}/stat")\n                if not stat.exists():\n                    break\n                try:\n                    if stat.read_text().rsplit(")", 1)[1].split()[0] == "Z":\n                        break\n                except FileNotFoundError:\n                    break\n                time.sleep(0.1)\n            else:\n                try:\n                    os.killpg(pid, signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n        file.unlink()\n    stop_app_processes()\n    for watch in STOP_WATCH:\n        watch.set()\n    for name in ("pairing.json", "stop-request.json", "stop-status.json"):\n        (PRIVATE / name).unlink(missing_ok=True)\n    for name in SESSION_KEYS:\n        config.pop(name, None)\n    private_write(CONFIG, json.dumps(config))\n    print(t("stop.done"))\n\n\ndef zip_outputs(destination="/content/comfy-output.zip"):\n    try:\n        with open(destination, "wb") as file:\n            app_action("zip-output", stdout=file, limit=1024 ** 4)\n    except Exception:\n        Path(destination).unlink(missing_ok=True)\n        raise\n    return str(destination)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("action", choices=["proxy", "supervise", "diagnose", "app"])\n    parser.add_argument("rest", nargs=argparse.REMAINDER)\n    args = parser.parse_args()\n    if args.action == "app":\n        if drops_privileges():\n            raise SystemExit("app actions must run as the application user")\n        BASE = Path(args.rest[0])\n        app_tree_action(args.rest[1], args.rest[2:], sys.stdin.buffer, sys.stdout.buffer)\n    elif args.action == "proxy":\n        from aiohttp import web\n        web.run_app(create_proxy(json.loads(CONFIG.read_text())), host="127.0.0.1", port=8189, access_log=None)\n    elif args.action == "diagnose":\n        asyncio.run(diagnose(json.loads(CONFIG.read_text())))\n    else:\n        supervise(json.loads(CONFIG.read_text()))\n', encoding='utf-8')

import importlib.util
spec = importlib.util.spec_from_file_location("comfy_colab_runtime", "/content/comfy_colab_runtime.py")
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
print("実行コードの展開が完了しました。")


In [ ]:
#@title 3. Driveを接続してセットアップ / 環境を復元
#@markdown 表示されるGoogleの画面でDriveへの接続を許可します。完了まで数分かかります。
runtime.setup(root_name=DRIVE_FOLDER_NAME, comfy_ref=COMFY_REF,
              checkpoint_seconds=CHECKPOINT_SECONDS, apt_packages=APT_PACKAGES.split(),
              restore=RESTORE_SAVED_ENVIRONMENT)
print("モデルの置き場所: マイドライブ/" + DRIVE_FOLDER_NAME + "/models/")


## モデル

Driveの `ComfyUI-Colab/models/` に、種類別のフォルダー（`checkpoints`・`diffusion_models`・`text_encoders`・`vae`・`loras`・`controlnet` など）があります。

- **手動で置く:** Driveの該当フォルダーへアップロードし、5を実行します。
- **URLから取得する:** 4にURLを貼り付けて実行します。Driveへ保存し、そのままColabにも反映します。


In [ ]:
#@title 4. URLからモデルを取得（Hugging Face / Civitai）
#@markdown Hugging Faceはファイルのページ（Files and versionsで開いたURL）、Civitaiはバージョンを選んだページかダウンロードリンクのURLを貼り付けます。
MODEL_URL = "" #@param {type:"string"}
#@markdown 保存先のフォルダー（一覧にない場合は入力できます）。
MODEL_FOLDER = "checkpoints" #@param ["checkpoints", "diffusion_models", "text_encoders", "vae", "loras", "controlnet", "clip_vision", "upscale_models", "embeddings"] {allow-input: true}
#@markdown 保存名（空欄なら配布元のファイル名）。
MODEL_FILENAME = "" #@param {type:"string"}
#@markdown 配布元が示すSHA256（任意。指定すると照合します）。
MODEL_SHA256 = "" #@param {type:"string"}
#@markdown 認証が必要なモデル: Colabのシークレット（鍵アイコン）に `HF_TOKEN` または `CIVITAI_TOKEN` を登録してONにします。
USE_DOWNLOAD_TOKEN = False #@param {type:"boolean"}
DOWNLOAD_NOW = False #@param {type:"boolean"}
if DOWNLOAD_NOW:
    config = runtime.json.loads(runtime.CONFIG.read_text())
    token = ""
    try:
        if USE_DOWNLOAD_TOKEN:
            from google.colab import userdata
            token = userdata.get(runtime.model_token_name(MODEL_URL))
        runtime.fetch_model(config, MODEL_URL, MODEL_FOLDER, MODEL_FILENAME, MODEL_SHA256, token)
    finally:
        token = ""
else:
    print("取得する場合は、URLを入力しDOWNLOAD_NOWをONにして実行してください。")


In [ ]:
#@title 5. Driveへ手動で置いたモデルを反映
REFRESH_MODELS = False #@param {type:"boolean"}
if REFRESH_MODELS:
    config = runtime.json.loads(runtime.CONFIG.read_text())
    runtime.sync_models(config)
    print("接続中の場合はComfyUIの画面を再読み込みしてください。同名のモデルを置き換えた場合は、10で停止（FLUSH_AND_UNMOUNTはOFF）してから6-1〜6-3のいずれかを実行します。")
else:
    print("Driveへ手動でモデルを追加したときに、REFRESH_MODELSをONにして実行してください。")


## 起動と接続

接続方法を1つ選んで実行します。別の方法へ切り替えるときは、10で停止（`FLUSH_AND_UNMOUNT` はOFF）してから実行します。初回の表示には1分前後かかることがあります。

- **6-1 プライベートURL:** このノートブックを開いているブラウザーで使う場合。このノートブックを実行しているGoogleアカウントでログインしたブラウザーからだけ開けます。外部には公開しません。Comfy Desktopからは接続できません。
- **6-2 公開URL:** Comfy Desktopの **Remote Connection** や別の端末から使う場合。表示されたURLを開き、ノートブックに表示されたコードを入力した端末だけが接続できます。コードは10分間・1回だけ有効で、URLは10で停止すると無効になります。
- **6-3 独自ドメインURL:** 自分のドメインの固定URLで、Comfy Desktopや別の端末から使う場合。Cloudflare Accessでログインした、許可済みのアカウントだけが接続できます。Cloudflareのアカウントとドメイン、事前の設定が必要です。
- Desktopでは、接続の設定で **Auto-download outputs** をONにすると生成物がPCへ保存されます。ブラウザーで使った場合や取得し損ねた出力は、9でまとめて回収できます。
- 画面が表示されない場合は、7の接続診断で止まっている箇所を確認できます。
- ComfyUIの画面上部の **停止** ボタンからも、環境を保存して接続を解除できます。そのままランタイムを削除することも選べます。


In [ ]:
#@title 6-1. 起動して接続: プライベートURL（このブラウザーのみ）
config = runtime.json.loads(runtime.CONFIG.read_text())
runtime.start(config, "colab")


In [ ]:
#@title 6-2. 起動して接続: 公開URL
#@markdown 6-2で起動した後、別の端末を追加する（またはコードの期限が切れた）ときにONにして実行すると、新しいペアリングコードを発行します。
NEW_PAIRING_CODE = False #@param {type:"boolean"}
config = runtime.json.loads(runtime.CONFIG.read_text())
if NEW_PAIRING_CODE:
    runtime.issue_pairing_code(config)
else:
    runtime.start(config, "tunnel")


In [ ]:
#@title 6-3. 起動して接続: 独自ドメインURL
#@markdown 事前にCloudflare Zero Trustで、名前付きトンネル（公開ホスト名の転送先は `http://127.0.0.1:8189`）と、そのホスト名を保護するAccessアプリケーションを作成します。Colabのシークレットに、トンネルのトークン（`eyJ` で始まる文字列）を `CF_TUNNEL_TOKEN`、AccessアプリケーションのApplication Audience (AUD) Tag（64桁）を `CF_ACCESS_AUD` の名前で登録し、このノートブックからのアクセスを許可してください。
#@markdown
#@markdown トンネルの公開ホスト名（例: `comfy.example.com`）。
ACCESS_HOSTNAME = "" #@param {type:"string"}
config = runtime.json.loads(runtime.CONFIG.read_text())
from google.colab import userdata
token, audience = userdata.get("CF_TUNNEL_TOKEN"), userdata.get("CF_ACCESS_AUD")
try:
    runtime.start_with_access(config, ACCESS_HOSTNAME, audience, token)
finally:
    token = audience = ""


In [ ]:
#@title 7. 保存状況とログ / 診断
#@markdown 画面の表示に必要な通信を、ComfyUI本体・認証プロキシ・公開URL（6-2のみ）の順に試します。
RUN_CONNECTION_DIAGNOSIS = False #@param {type:"boolean"}
#@markdown ComfyUI用のユーザーから届くローカルサービスを一覧表示します。
RUN_ISOLATION_CHECK = False #@param {type:"boolean"}
if RUN_CONNECTION_DIAGNOSIS:
    runtime.run([runtime.sys.executable, "/content/comfy_colab_runtime.py", "diagnose"], timeout=600)
if RUN_ISOLATION_CHECK:
    runtime.isolation_report()
path = runtime.PRIVATE / "checkpoint-status.json"
print(path.read_text() if path.exists() else "保存状況はまだありません。")
for name in ("comfy.log", "supervisor.log", "proxy.log", "tunnel.log"):
    path = runtime.PRIVATE / name
    if path.exists():
        print("---", name)
        print(runtime.redact(path.read_text(errors="replace")[-5000:]))


## 保存と終了

環境（ComfyUI本体・カスタムノード・設定・依存関係の一覧）は自動でDriveへ保存され、次回の3で復元されます。直前の変更を確実に残したいときは8を使います。生成物はDriveへ保存されません。

終了するときは、PCへの出力の取得を確認してから10を実行し、完了の表示を確認してからランタイムを削除してください。


In [ ]:
#@title 8. 環境を今すぐDriveへ保存
SAVE_NOW = False #@param {type:"boolean"}
if SAVE_NOW:
    config = runtime.json.loads(runtime.CONFIG.read_text())
    runtime.checkpoint(config, force=True)
    print("環境と新しいモデルをDriveへ保存しました。")
else:
    print("保存する場合は、SAVE_NOWをONにして実行してください。")


In [ ]:
#@title 9. 生成物をZIPでPCへダウンロード
DOWNLOAD_OUTPUT_ZIP = False #@param {type:"boolean"}
if DOWNLOAD_OUTPUT_ZIP:
    from google.colab import files
    files.download(runtime.zip_outputs())
else:
    print("回収する場合は、DOWNLOAD_OUTPUT_ZIPをONにして実行してください。")


In [ ]:
#@title 10. 保存して停止
STOP_NOW = False #@param {type:"boolean"}
#@markdown Driveへの書き込みを完了して接続を解除する（ランタイムを削除する前はON。同じランタイムで6-1〜6-3のいずれかから再開するならOFF）。
FLUSH_AND_UNMOUNT = True #@param {type:"boolean"}
if STOP_NOW:
    config = runtime.json.loads(runtime.CONFIG.read_text())
    runtime.stop(config, save=True)
    if FLUSH_AND_UNMOUNT:
        from google.colab import drive
        drive.flush_and_unmount()
        print("Driveへの書き込みが完了しました。ランタイムを削除できます。")
    else:
        print("Driveは接続したままです。ランタイムを削除する前に、FLUSH_AND_UNMOUNTをONにして実行してください。")
else:
    print("終了する場合は、STOP_NOWをONにして実行してください。")


## ComfyUIの更新

保存した環境を復元すると、ComfyUIは保存したときのバージョンのままです。カスタムノード・設定・ワークフローを残したままComfyUI本体を更新するには11を実行します。

- **手順:** 0をONにして実行（または1の `COMFY_REF` に更新先のバージョンを入力）→ 1 → 2 → 3 → 11 → 6-1〜6-3 のいずれか。起動中の場合は、10で停止（`FLUSH_AND_UNMOUNT` はOFF）してから 0 → 1 → 11 → 6-1〜6-3 のいずれか を実行します。
- 更新後の環境はDriveへ保存され、次回からは新しいバージョンで復元されます。
- 更新に失敗した場合（新しいバージョンとカスタムノードの依存関係が合わない場合など）は元のバージョンへ戻し、Driveの保存内容は変えません。


In [ ]:
#@title 11. ComfyUIを更新（カスタムノードと設定は維持）
#@markdown `COMFY_REF`（0で最新リリースにした場合はそのバージョン）へ更新します。ComfyUIを停止した状態で実行してください。
UPDATE_COMFYUI_NOW = False #@param {type:"boolean"}
if UPDATE_COMFYUI_NOW:
    config = runtime.json.loads(runtime.CONFIG.read_text())
    runtime.update_comfyui(config, COMFY_REF)
else:
    print("更新する場合は、UPDATE_COMFYUI_NOWをONにして実行してください。")


## セキュリティについて

- 6-1は外部に公開しません。6-2と6-3で公開されるのは認証付きの入口だけです。6-2ではペアリングコードを入力していない端末、6-3ではCloudflare Accessの署名がない要求を拒否し、ComfyUI本体は外部から直接は見えません。トンネルには [Cloudflare Tunnel](https://developers.cloudflare.com/tunnel/) を使います（6-2は [Quick Tunnel](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)）。
- ComfyUIとカスタムノードは権限を下げた専用ユーザーで動き、Driveのマウント先と接続用の秘密情報には届きません。ただしコンテナーによる隔離ではありません。Colab自身の管理用サービスへはこのユーザーからも接続できるため、Colabを狙った悪意あるノードまでは防げません。
- 接続できた利用者はカスタムノードを導入でき、保存した環境に含まれるノードは次回も実行されます。不審なノードを入れてしまった場合は、1の `RESTORE_SAVED_ENVIRONMENT` をOFFにして新しい環境で始めてください。
- Colabの「Driveをマウント」ボタンは使わず、このノートブックのセットアップだけでDriveへ接続してください。
